In [1]:
!pip install monai

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 13.2 MB/s eta 0:00:0000:010:01


In [ ]:
# %% [markdown CELL 1] ========================================================
# N3 - interactive certified report for one case (CPU)
#
# Build the interactive report for one case: slice viewer with the error map, the
# certified volume interval, the risk curves and the coverage table.  CPU only, and it reads
# only files N2 wrote, so it can be re-run as often as you like.
#
# Each case is reported against the certificate calibrated on ITS OWN contrast stratum when
# that subgroup is certifiable (safepp.choose_certificate decides; the pooled certificate is
# the fallback), so a low-contrast scan carries the tighter threshold and the subgroup's own
# evidence rather than the cohort average.

# %% [CELL 2] SETTINGS -------------------------------------------------------
CASE_ID     = None         # None = auto-pick the most instructive test case
SPLIT       = "test"       # test | val
ARM         = "base"       # fallback only: the body now picks the first entry of
                           # ARM_PREFERENCE that has a certificate, so the delivered
                           # configuration (e.g. the ensemble) is what gets shown
ARM_PREFERENCE = ["base", "ens", "sr50"]   # [final 2026-10-10] base delivered (ensemble gave no measurable gain); must match the arm you present
PRIMARY     = "wt"         # region the certificate is about (wt | tc | et); N2
                           # records its own choice in certificates.json and that
                           # one wins if they disagree
STRATIFIED  = True         # report each case against the certificate calibrated on ITS OWN
                           # contrast stratum (low-contrast scans get the tighter threshold and
                           # the subgroup's evidence).  Falls back to the pooled certificate
                           # when the subgroup is not certifiable or the contrast is unknown --
                           # safepp.choose_certificate decides, and the note says which was used.
N_REPORT    = 37            # how many cases to render (each becomes its own HTML file)
WORK        = "/kaggle/working"
INPUT_ROOT  = "/kaggle/input"
CACHE_BASE  = None         # None = auto-discover the attached cache shard(s)
PROBS_DIR   = None         # None = auto-discover the attached probs dataset
VIEW3D_N     = 96           # coarse grid for the 3D viewer and the live-λ preview.  The coarse
                            # volumes go into the page as grayscale PNGs (a probability field
                            # run-length codes badly), so this trades page size against
                            # smoothness -- 64 lands a one-case report near 1.5-2 MB, which opens
                            # instantly from disk.  Every NUMBER in the report is full resolution.
ORTHO_SLICES = 28            # background slices shipped per cross-plane stack (coronal/sagittal)
N_SLICES    = 64           # axial slices kept per case (memory) -- the tumour band is centred
PLOTLY_BUNDLE = "inline"    # how the interactive 3D viewer gets plotly.js:
                            #   "inline" -> ~6.6 MB embedded in the report (and in the notebook's
                            #               own output).  Self-contained: no CDN, no network, opens
                            #               offline, survives a locked-down preview pane.
                            #   "cdn"    -> ~3 MB page that fetches plotly.js when it is opened.
                            #               Use it for a smaller file when you know the machine is
                            #               online; the page falls back to its own canvas renderer
                            #               if the fetch fails.
                            # Kaggle's limits: ~1 MiB on the notebook FILE (so never paste a
                            # library into a cell) but ~100 MiB on file + outputs, which is where
                            # this bundle legitimately lives.


# ---------------------------------------------------------------- build fingerprint
BUILD = "N3 2026-10-10 final: base delivered, size-aware lesion wording, low-contrast evidence, page .nii fix, outline-based description (+ FIX v2.1 cell 91)"
print(f"BUILD: {BUILD}")
print(f"  expected fixes in this copy: self-contained HTML (no CDN, no network) | follows the frozen certificate | coverage table alongside the curves | cockpit layout, all-plane centring, working post-processing toggle, region-coloured 3D")
print("  If you do NOT see this BUILD line in the output, you are running an older copy of the "
      "notebook: re-import "
      f"N3_report.ipynb from kaggle_v2_scripts.zip (File -> Import Notebook) instead of hand-patching "
      "cells.  Half of one build and half of another is what makes a fix look like it did not work.")

# %% [CELL 3] SHARED LIBRARY (bratsx + safepp) -- identical in every notebook -----
from __future__ import annotations

# ============================================================================
# bratsx.py -- shared library for the BraTS2020 x fractional-DoD study
# (attached to each notebook as a single "SHARED LIBRARY" cell)
#
# Contents
#   A. fractional operators  : riesz_weights, RieszConv3d, FractionalOrderMixture,
#                              spectral_frac_laplacian, gaussian_blur3d
#   B. prior head            : FracPrior (frozen-DoD | learnable-mixture | DoG)
#   C. model zoo             : SegResNetRefine (control), FracPriorNet (arms)
#   D. data pipeline         : decode_*, PatchDataset, sliding-window inference
#   E. metrics               : dice_np, hd95_fast, band_dice, postprocess
#   F. kaggle plumbing       : session budget, run-queue markers, disk guards
# ============================================================================
import glob, hashlib, json, math, os, shutil, time
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

MODS = ["flair", "t1", "t1ce", "t2"]
DOD_CH = (0, 2)                      # FLAIR, T1ce -- the modalities the DoD arms consume
FULL_VAL_CASES = 37                  # the fixed split has 37 val / 37 test


# ===========================================================================
# A. fractional operators
# ===========================================================================
def _sgn_gamma(x: torch.Tensor) -> torch.Tensor:
    return torch.where(x > 0, torch.ones_like(x), (-1.0) ** (torch.floor(-x) + 1))


def riesz_weights(alpha: torch.Tensor, K: int = 3) -> torch.Tensor:
    """Riesz / fractional centred-difference taps, differentiable in alpha, zero-sum.
    alpha = 2 -> [1,-2,1].  Returns (C, 2K+1)."""
    a = alpha.reshape(-1).to(torch.float64)
    ks = torch.arange(-K, K + 1, dtype=torch.float64, device=a.device)
    x0, xa, xb = a + 1.0, a[:, None] / 2 - ks[None, :] + 1, a[:, None] / 2 + ks[None, :] + 1
    logabs = torch.lgamma(x0)[:, None] - torch.lgamma(xa) - torch.lgamma(xb)
    sgn = _sgn_gamma(x0)[:, None] * _sgn_gamma(xa) * _sgn_gamma(xb) * (-((-1.0) ** ks))
    w = sgn * torch.exp(logabs)
    pole = ((xa <= 0) & (xa == xa.round())) | ((xb <= 0) & (xb == xb.round()))
    w = torch.where(pole, torch.zeros_like(w), w).clone()
    w[:, K] = w[:, K] - w.sum(dim=1)
    return torch.nan_to_num(w, nan=0.0).to(alpha.dtype)


def riesz_weights_np(alpha: float, K: int = 3):
    """numpy twin, for data-side use (matches the reference implementation)."""
    from scipy.special import gamma, rgamma
    ks = np.arange(-K, K + 1)
    g = (-1.0) ** ks * gamma(alpha + 1) * rgamma(alpha / 2 - ks + 1) * rgamma(alpha / 2 + ks + 1)
    w = -g
    w[K] -= w.sum()
    return w, ks


def _conv1d_axis(x, w, K, dim, groups):
    pad = [0] * 6
    pad[2 * (2 - dim)] = K
    pad[2 * (2 - dim) + 1] = K
    xp = F.pad(x, pad, mode="replicate")
    C = x.shape[1]
    shape = [1, 1, 1, 1, 1]
    shape[2 + dim] = 2 * K + 1
    return F.conv3d(xp, w.reshape(shape).expand(C, 1, *shape[2:]).to(x.dtype), groups=C)


class FractionalOrderMixture(nn.Module):
    """Softmax mixture over a fixed grid of orders.  Returns Delta^alpha x with
    E[alpha] = sum_a p_a alpha_a -- the number to report.  Numerically bulletproof
    (no Gamma poles, unlike a continuous alpha)."""

    def __init__(self, alphas=(0.8, 1.0, 1.2, 1.5, 1.8, 2.0), K=3, temperature=1.0,
                 init_probs=None):
        super().__init__()
        self.K = K
        self.register_buffer("alphas", torch.tensor(alphas, dtype=torch.float32))
        self.register_buffer("kernels", torch.stack(
            [torch.tensor(riesz_weights_np(float(a), K)[0], dtype=torch.float32) for a in alphas]))
        self.temperature = temperature
        init = torch.ones(len(alphas)) if init_probs is None else torch.tensor(np.asarray(init_probs), dtype=torch.float32)
        self.logits = nn.Parameter(torch.log(init / init.sum()))

    def posterior(self):
        return torch.softmax(self.logits / self.temperature, dim=0)

    def mean_alpha(self):
        return (self.posterior() * self.alphas).sum()

    def forward(self, x):
        w = self.posterior() @ self.kernels
        return sum(_conv1d_axis(x, w, self.K, d, x.shape[1]) for d in range(3))

    def extra_repr(self):
        return f"E[alpha]={float(self.mean_alpha()):.3f} p={[round(float(v),3) for v in self.posterior()]}"


class RieszConv3d(nn.Module):
    def __init__(self, channels, K=3, alpha_init=1.5, per_channel=True, lo=0.2, hi=2.0):
        super().__init__()
        self.K, self.channels, self.per_channel, self.lo, self.hi = K, channels, per_channel, lo, hi
        n = channels if per_channel else 1
        u = (alpha_init - lo) / (hi - lo)
        self.alpha_raw = nn.Parameter(torch.full((n,), math.log(u / (1 - u))))

    @property
    def alpha(self):
        a = self.lo + (self.hi - self.lo) * torch.sigmoid(self.alpha_raw)
        return a if self.per_channel else a.expand(self.channels)

    def forward(self, x):
        w = riesz_weights(self.alpha, self.K)[:, None, :].to(x.dtype)     # (C,1,2K+1)
        return sum(_conv1d_axis(x, w, self.K, d, x.shape[1]) for d in range(3))


def _frac_symbol(shape, alpha, radial=True, device="cpu"):
    fs = [torch.fft.fftfreq(n, device=device, dtype=torch.float32) * 2 * math.pi for n in shape]
    g = torch.meshgrid(*fs, indexing="ij")
    if radial:
        return -((sum(4.0 * torch.sin(x / 2.0) ** 2 for x in g)) + 1e-12) ** (alpha / 2.0)
    return -sum((4.0 * torch.sin(x / 2.0) ** 2 + 1e-12) ** (alpha / 2.0) for x in g)


def spectral_frac_laplacian(x, alpha, radial=True):
    S = _frac_symbol(x.shape[-3:], alpha, radial, x.device)
    X = torch.fft.fftn(x.float(), dim=(-3, -2, -1))
    return torch.fft.ifftn(X * S, dim=(-3, -2, -1)).real.to(x.dtype)


def gaussian_blur3d(x, sigma, truncate=3.0):
    """separable Gaussian blur, groups=C (the classical anchor for the DoG arm)"""
    if sigma <= 0:
        return x
    r = int(truncate * sigma + 0.5)
    k = torch.exp(-0.5 * (torch.arange(-r, r + 1, device=x.device, dtype=torch.float32) / sigma) ** 2)
    k = (k / k.sum()).to(x.dtype)
    C = x.shape[1]
    out = x
    for d in range(3):
        pad = [0] * 6
        pad[2 * (2 - d)] = r
        pad[2 * (2 - d) + 1] = r
        shape = [1, 1, 1, 1, 1]
        shape[2 + d] = 2 * r + 1
        out = F.conv3d(F.pad(out, pad, mode="replicate"),
                       k.reshape(shape).expand(C, 1, *shape[2:]), groups=C)
    return out


def spectral_dod_volume(x, alpha, T1=4.0, T2=12.0, radial=True, pad=24,
                        signed=False, smooth_sigma=0.0):
    """Exact DoD of one volume, x = (1,1,D,H,W) or (D,H,W) float tensor.
    smooth_sigma > 0 pre-smooths (stand-in for the papers' RBF interpolant).
    Returns |DoD|, or the tuple (|DoD|, signed DoD) when signed=True."""
    u = x.float()[None, None] if x.dim() == 3 else x.float()
    if smooth_sigma and smooth_sigma > 0:
        from scipy.ndimage import gaussian_filter
        u = torch.from_numpy(gaussian_filter(u[0, 0].numpy(), smooth_sigma, mode="nearest"))[None, None]
    up = F.pad(u, [pad] * 6, mode="replicate") if pad else u
    S = _frac_symbol(up.shape[-3:], alpha, radial, "cpu")
    H = (torch.exp(T2 * S) - torch.exp(T1 * S)).float()
    y = torch.fft.ifftn(torch.fft.fftn(up) * H, dim=(-3, -2, -1)).real
    if pad:
        y = y[..., pad:-pad, pad:-pad, pad:-pad]
    return (y.abs(), y) if signed else y.abs()


def match_band_T(alpha, ref_alpha=2.0, ref_T1=4.0, ratio=3.0, radial=True):
    """T1 such that the DoD band centre of `alpha` coincides with the reference
    (alpha=2, T1=4) band centre.  alpha=1.5 -> (2.435, 7.305).  radius=3."""
    k = 2.0 * math.pi / 15.0                      # band centre used by the reference
    ref_sigma = (2.0 * k ** 2) ** (ref_alpha / 2.0) / 2.0 if radial else k ** ref_alpha / 2.0
    sigma = (math.log(ratio) / (ref_T1 * (ratio - 1.0))) if ref_T1 else None
    # numerical solve is more robust than the closed form above
    def peak_w(a, T1):
        ww = np.linspace(1e-4, np.pi, 4000)
        S = -(ww ** a) if not radial else -((2 - 2 * np.cos(ww)) ** (a / 2.0))
        R = np.abs(np.exp(3 * T1 * S) - np.exp(T1 * S))
        return ww[int(np.argmax(R))]
    target = peak_w(ref_alpha, ref_T1)
    lo, hi = 1e-3, 60.0
    for _ in range(60):
        mid = 0.5 * (lo + hi)
        if peak_w(alpha, mid) > target:
            lo = mid
        else:
            hi = mid
    T1 = 0.5 * (lo + hi)
    return float(T1), float(T1 * ratio)


# ===========================================================================
# B. prior head -- the operator's only route into the network
# ===========================================================================
class FracPrior(nn.Module):
    """logits_out = logits_in + sum_m beta[c,m] * R(x)[:, m]

    mode="frozen": R = identity on the precomputed DoD channels (x[:, 4:])
    mode="mix"   : R = Delta^{alpha} with a Learnable Mixture over orders, on x[:, :4]
    mode="dog"   : R = x - G_sigma*x  (classical high-pass anchor), on x[:, :4]

    beta is zero-initialised => at step 0 the arm is BIT-IDENTICAL to the control.
    """

    def __init__(self, n_classes=3, n_ch=4, mode="mix", K=3, alpha_init=1.5,
                 dog_sigma=2.0, alphas=(0.8, 1.0, 1.2, 1.5, 1.8, 2.0), init_at_alpha2=True):
        super().__init__()
        self.mode, self.n_ch, self.K = mode, n_ch, K
        self.dog_sigma = dog_sigma
        if mode == "mix":
            init = np.ones(len(alphas))
            if init_at_alpha2:
                init[np.argmin([abs(a - 2.0) for a in alphas])] = 4.0   # start near classical
            self.op = FractionalOrderMixture(alphas=alphas, K=K, init_probs=init)
        else:
            self.op = None
        self.beta = nn.Parameter(torch.zeros(n_classes, n_ch))

    def alpha_value(self):
        return float(self.op.mean_alpha()) if self.op is not None else float("nan")

    def operator(self, x):
        mri = x[:, :4]
        if self.mode == "mix":
            return self.op(mri)
        if self.mode == "dog":
            return mri - gaussian_blur3d(mri, self.dog_sigma)
        return x[:, 4:4 + self.n_ch]                     # frozen precomputed DoD

    def forward(self, logits, x):
        r = self.operator(x)
        s = r.flatten(2).std(dim=2).clamp_min(1e-6).view(r.shape[0], -1, 1, 1, 1)
        r = torch.clamp(r / s, -6.0, 6.0)
        return logits + torch.einsum("cm,bm...->bc...", self.beta, r)

    def extra_repr(self):
        return f"mode={self.mode} n_ch={self.n_ch} " + \
               (f"E[alpha]={self.alpha_value():.3f}" if self.op is not None else "")


# ===========================================================================
# C. model zoo
# ===========================================================================
try:                                                  # noqa: E402
    from monai.networks.nets import SegResNet
    from monai.networks.blocks import Convolution
    from monai.losses import DiceLoss
except ImportError as _e:                            # pragma: no cover
    raise ImportError(
        "MONAI is not installed in this image. Fix it in one of two ways, then re-run from the "
        "top: (a) notebook Settings -> Internet = On and run  !pip install monai  in a cell above; "
        "or (b) use the notebook's Dependency Manager to add 'monai'. Kaggle's standard image has "
        "monai, torch, nibabel, scipy, pandas and scikit-learn pre-installed."
    ) from _e


def _segresnet():
    return SegResNet(spatial_dims=3, in_channels=4, out_channels=3, init_filters=16,
                     blocks_down=(1, 2, 2, 4), blocks_up=(1, 1, 1), dropout_prob=0.2)


class RefineHead(nn.Module):
    """capacity-matched refinement head: sees [coarse probs, input channels]"""

    def __init__(self, in_ch):
        super().__init__()
        last = nn.Conv3d(16, 3, 1)
        nn.init.zeros_(last.weight); nn.init.zeros_(last.bias)
        self.net = nn.Sequential(
            Convolution(spatial_dims=3, in_channels=3 + in_ch, out_channels=16, kernel_size=3,
                        strides=1, norm="instance", act="relu"),
            Convolution(spatial_dims=3, in_channels=16, out_channels=16, kernel_size=3,
                        strides=1, norm="instance", act="relu"),
            last)

    def forward(self, coarse, x):
        return self.net(torch.cat([torch.sigmoid(coarse), x], dim=1))


class FracPriorNet(nn.Module):
    """backbone (MRI only) + refine head + optional operator prior.
    mode=None reproduces the capacity-matched control exactly."""

    def __init__(self, in_ch=4, prior_mode=None, alpha_init=1.5, n_prior_ch=4, dog_sigma=2.0):
        super().__init__()
        self.in_ch = in_ch
        self.backbone = _segresnet()
        self.head = RefineHead(in_ch)
        self.prior = None if prior_mode is None else FracPrior(
            n_classes=3, n_ch=n_prior_ch, mode=prior_mode, alpha_init=alpha_init, dog_sigma=dog_sigma)

    def forward(self, x):
        coarse = self.backbone(x[:, :4])
        logits = coarse + self.head(coarse, x)
        if self.prior is not None:
            logits = self.prior(logits, x)
        return logits

    def alpha_value(self):
        return float("nan") if self.prior is None else self.prior.alpha_value()


def build_model(arm: str, alpha=1.5):
    """arm names used across the notebooks"""
    if arm == "mri":
        return _segresnet()                                     # baseline A
    if arm == "mri_ref":
        return FracPriorNet(in_ch=4, prior_mode=None)            # control B (capacity matched)
    if arm == "mri_ref_dod":
        # CONTROL for the extra channels: same 6-ch input, same head, prior switched OFF.
        # dod15 vs mri_ref_dod therefore differs by beta*R ONLY -> bit-identical at init.
        return FracPriorNet(in_ch=4 + len(DOD_CH), prior_mode=None)
    if arm in ("dod15", "dod20", "dod_shuf"):
        return FracPriorNet(in_ch=4 + len(DOD_CH), prior_mode="frozen", n_prior_ch=len(DOD_CH))
    if arm == "dod_fit":
        return FracPriorNet(in_ch=4, prior_mode="mix")
    if arm == "dod_dog":
        return FracPriorNet(in_ch=4, prior_mode="dog")
    raise ValueError(arm)


ALL_ARMS = ("mri", "mri_ref", "mri_ref_dod", "dod15", "dod20", "dod_fit", "dod_dog", "dod_shuf")
DOD_ARMS = ("mri_ref_dod", "dod15", "dod20", "dod_shuf")


def n_input_channels(arm):
    return 4 + (len(DOD_CH) if arm in DOD_ARMS else 0)


# ===========================================================================
# D. data pipeline
# ===========================================================================
def load_nii(path):
    import nibabel as nib
    return np.nan_to_num(nib.load(path).get_fdata(dtype=np.float32), nan=0.0, posinf=0.0, neginf=0.0)


def list_candidates(pdir, pid, kind, deep=True):
    """Every plausible path for one volume of one case, BEST GUESS FIRST.

    Listing-based and case-INsensitive on purpose.  Real mirrors rename things in ways that
    glob patterns miss: `..._Seg.nii` (capital S), `..._SEG.nii`, `segmentation.nii`,
    `..._seg.nii/..._seg.nii` (the "file" is a folder), a stray suffix, or the bare `seg.nii`.
    A naive `glob(pdir + "*seg*")` returns nothing for the first two, which is what turned a
    single odd case into a dead worker pool (the exception surfaces as IndexError, not as a
    missing-file message).
    """
    exact, base, kind_pid, kind_only = [], [], [], []
    kl, pl = kind.lower(), pid.lower()
    for e in ("nii.gz", "nii", "nii.z", "nii.gz.z", ""):
        nm = f"{pid}_{kind}" + (f".{e}" if e else "")
        for variant in {nm, nm.lower(), nm.upper()}:
            exact.append(os.path.join(pdir, variant))

    def scan(d):
        try:
            entries = sorted(os.listdir(d))
        except OSError:
            return
        for nm in entries:
            full = os.path.join(d, nm)
            if os.path.isdir(full):
                if deep:
                    scan(full)                       # the "file" may be a folder holding it
                continue
            low = nm.lower()
            if low.startswith(pl) and low.startswith(f"{pl}_{kl}"):
                base.append(full)
            elif pl in low and kl in low:
                kind_pid.append(full)
            elif kl in low:
                kind_only.append(full)

    scan(pdir)
    out = []
    for group in (exact, base, kind_pid, kind_only):
        for q in group:
            if os.path.isfile(q) and q not in out:
                out.append(q)
    return out


def case_file(pdir, pid, kind, deep=True):
    """The path to hand to nibabel, or None.  nib.load sniffs the content, so a gzip file
    named `.nii` still loads."""
    cands = list_candidates(pdir, pid, kind, deep=deep)
    return cands[0] if cands else None


def describe_case(pdir, limit=12, kind=None):
    """One-line diagnostic for error messages: what is actually in the case folder (and, for
    the requested volume, the near-misses that were rejected)."""
    try:
        entries = sorted(os.listdir(pdir))
    except OSError as e:
        return f"(cannot list {pdir}: {e})"
    head = entries[:limit]
    more = "" if len(entries) <= limit else f" … +{len(entries) - limit} more"
    txt = f"{pdir} contains {len(entries)} entries: {head}{more}"
    if kind:
        near = [q for q in list_candidates(pdir, pid_of_dir(pdir), kind)
                if os.path.basename(q).lower().find(kind.lower()) >= 0]
        txt += f"\n  names containing {kind!r}: {[os.path.basename(q) for q in near] or 'NONE'}"
        sub = [q for q in sorted(glob.glob(os.path.join(pdir, "**", "*"), recursive=True))
               if os.path.isdir(q)][:5]
        if sub:
            txt += f"\n  subfolders: {[os.path.relpath(q, pdir) for q in sub]}"
    return txt


def pid_of_dir(pdir):
    b = os.path.basename(pdir.rstrip("/"))
    return b if b.startswith("BraTS") else "BraTS20_Training_000"


def find_file(pdir, pid, kind):
    p = case_file(pdir, pid, kind)
    if p is None:
        raise FileNotFoundError(
            f"no '{kind}' volume for {pid}\n  {describe_case(pdir, kind=kind)}\n"
            f"  Accepted: '{pid}_{kind}.nii(.gz)' in any case or spelling, optionally inside a "
            f"folder of the same name, or any file whose name contains '{kind}'.")
    return p


def find_seg(pdir, pid):
    return find_file(pdir, pid, "seg")


def load_case_volumes(pdir, pid, mods=None):
    """(mri_stack, seg) for one case, resolved against whatever the mirror named the files.

    This is what the prep notebook calls: it never constructs a filename by hand, so a
    `.nii`-only mirror, a wrapped-in-folder mirror or an uppercase mirror all work.
    """
    mods = list(mods or MODS)
    vols = [load_nii(find_file(pdir, pid, m)) for m in mods]
    seg = np.rint(load_nii(find_file(pdir, pid, "seg"))).astype(np.uint8)
    seg[seg == 3] = 1                                  # BraTS label convention
    return np.stack(vols), seg


def normalize_mri_single(vol):
    """z-score inside the non-zero mask, clip to the 0.5-99.5 percentile first."""
    m = np.abs(vol) > 1e-6
    out = np.zeros(vol.shape, np.float32)
    if m.sum() < 10:
        return out
    lo, hi = np.percentile(vol[m], [0.5, 99.5])
    v = np.clip(vol[m], lo, hi)
    out[m] = (v - v.mean()) / (v.std() + 1e-8)
    return out


def as_roots(root):
    """A cache may be one dir, a LIST of dirs (Kaggle shards), or a glob pattern."""
    if root is None:
        return []
    items = list(root) if isinstance(root, (list, tuple, set)) else [root]
    out = []
    for r in items:
        cands = sorted(glob.glob(str(r))) if any(c in str(r) for c in "*?[") else [str(r)]
        for c in cands:
            if os.path.isdir(c) and c not in out:
                out.append(c)
    return out


def _first_file(roots, name):
    """the first shard that actually contains `name` (loaders never listdir)."""
    for r in roots:
        f = os.path.join(r, name)
        if os.path.exists(f):
            return f
    return os.path.join(roots[0], name) if roots else name


def cache_file(cache, pid, suffix, ext="npy"):
    return _first_file(as_roots(cache), f"{pid}_{suffix}.{ext}")


def has_case(cache, pid):
    return os.path.exists(cache_file(cache, pid, "mri"))


def loader_case(cache, pid, dod_tag=None, dod_root=None):
    """-> (x float32 (C,D,H,W), seg uint8, dist float32).  Every channel is unit scale."""
    c = as_roots(cache)
    dr = as_roots(dod_root) or c
    x = np.load(_first_file(c, f"{pid}_mri.npy")).astype(np.float32)
    if dod_tag:
        d = np.load(_first_file(dr, f"{pid}_{dod_tag}.npy")).astype(np.float32)
        x = np.concatenate([x, d], axis=0)
    return x, np.load(_first_file(c, f"{pid}_seg.npy")), \
        decode_dist(np.load(_first_file(c, f"{pid}_dist.npy")))


def dod_tag(alpha):
    """canonical file tag for a precomputed DoD: 1.5 -> 'dod_a15', 2.0 -> 'dod_a20'"""
    return f"dod_a{int(round(float(alpha) * 10)):02d}"


def decode_dod(u16, range_=32.0):
    return (u16.astype(np.float32) / 65535.0) * range_


def bbox_crop_slices(mri, margin=4):
    brain = (np.abs(mri) > 1e-6).any(axis=0)
    idx = np.where(brain)
    if len(idx[0]) == 0:
        return tuple(slice(0, s) for s in brain.shape)
    return tuple(slice(max(0, int(idx[a].min()) - margin),
                       min(brain.shape[a], int(idx[a].max()) + 1 + margin)) for a in range(3))


def normalize_mri(mri):
    out = np.zeros(mri.shape, dtype=np.float32)
    for c in range(mri.shape[0]):
        m = np.abs(mri[c]) > 1e-6
        if m.sum() < 10:
            continue
        lo, hi = np.percentile(mri[c][m], [0.5, 99.5])
        v = np.clip(mri[c][m], lo, hi)
        out[c][m] = (v - v.mean()) / (v.std() + 1e-8)
    return out


def seg_to_targets(seg):
    return np.stack([seg > 0, (seg == 1) | (seg == 4), seg == 4]).astype(np.float32)


def signed_distance_to_wt(seg, clip=40.0):
    from scipy.ndimage import distance_transform_edt
    wt = seg > 0
    if not wt.any():
        return np.zeros(seg.shape, np.float32)
    return np.clip(distance_transform_edt(~wt) - distance_transform_edt(wt), -clip, clip).astype(np.float32)


def encode_dist(d, clip=40.0):
    """signed distance -> uint8 (4.8 MB/case instead of 9.6).  80 voxels / 255 levels
    = 0.31 voxel resolution, which is far finer than the tau=3 weighting needs."""
    z = np.clip((np.asarray(d, np.float32) + clip) / (2 * clip), 0, 1)
    return np.rint(z * 255).astype(np.uint8)


def decode_dist(u, clip=40.0):
    return (np.asarray(u, np.float32) / 255.0) * (2 * clip) - clip


def sample_locs(seg, n=3000, seed=0):
    rng = np.random.default_rng(seed)
    d = {}
    for name, cond in (("c1", seg == 1), ("c2", seg == 2), ("c4", seg == 4)):
        co = np.argwhere(cond)
        if len(co) > n:
            co = co[rng.choice(len(co), n, replace=False)]
        d[name] = co.astype(np.int16)
    return d


class PatchDataset(torch.utils.data.Dataset):
    """Random patch sampler.  Sample i is a pure function of (SEED, offset+i) so every
    arm sees the identical sample sequence -> the comparison is paired by construction.

    Returns (x[C,D,H,W], y[3,D,H,W], dist[1,D,H,W]).  x channels are ALL unit scale.
    """

    P_FG = 0.70

    def __init__(self, cache, ids, arm, length, offset=0, seed=42, patch=(96, 96, 96),
                 dod_tag=None, dod_root=None, shuffle_dod=False, jitter=True):
        self.cache, self.ids, self.arm = as_roots(cache), list(ids), arm
        self.length, self.offset, self.seed, self.patch = length, offset, seed, tuple(patch)
        # dod_tag: which precomputed DoD to load, e.g. "dod_a15" / "dod_a20". None = no DoD.
        # dod_root: the DoD is a SEPARATE dataset from the MRI cache on Kaggle.
        self.dod_tag, self.shuffle_dod, self.jitter = dod_tag, shuffle_dod, jitter
        self.dod_root = as_roots(dod_root) or self.cache
        self.use_dod = dod_tag is not None

    def __len__(self):
        return self.length

    def _p(self, pid, s, ext="npy"):
        return _first_file(self.cache, f"{pid}_{s}.{ext}")

    def __getitem__(self, i):
        rng = np.random.default_rng([self.seed, self.offset + i])
        pid = self.ids[int(rng.integers(len(self.ids)))]
        mri = np.load(self._p(pid, "mri"), mmap_mode="r")
        seg = np.load(self._p(pid, "seg"), mmap_mode="r")
        dist = np.load(self._p(pid, "dist"), mmap_mode="r")
        shape = seg.shape

        center = None
        if rng.random() < self.P_FG:
            locs = np.load(self._p(pid, "locs", "npz"))
            present = [k for k in ("c1", "c2", "c4") if len(locs[k]) > 0]
            if present:
                k = present[int(rng.integers(len(present)))]
                center = locs[k][int(rng.integers(len(locs[k])))]
        if center is None:
            center = [rng.integers(s) for s in shape]
        st = [int(np.clip(c - p // 2, 0, max(0, s - p))) for c, s, p in zip(center, shape, self.patch)]
        sl = tuple(slice(a, a + p) for a, p in zip(st, self.patch))

        x = np.asarray(mri[(slice(None),) + sl], dtype=np.float32)
        if self.use_dod:
            src = pid
            if self.shuffle_dod:                      # placebo arm: DoD from ANOTHER case
                src = self.ids[int(rng.integers(len(self.ids)))]
            d = np.load(_first_file(self.dod_root, f"{src}_{self.dod_tag}.npy"), mmap_mode="r")
            d = np.stack([np.asarray(d[c][sl], dtype=np.float32) for c in range(d.shape[0])])
            p = [(0, 0)] + [(0, q - t) for q, t in zip(self.patch, d.shape[1:])]
            if any(b for _, b in p[1:]):
                d = np.pad(d, p)
            x = np.concatenate([x, d], axis=0)
        y = seg_to_targets(np.asarray(seg[sl]))
        dd = decode_dist(np.asarray(dist[sl]))[None]

        pad = [(0, 0)] + [(0, p - s) for p, s in zip(self.patch, x.shape[1:])]
        if any(b for _, b in pad):
            x = np.pad(x, pad)
            y = np.pad(y, [(0, 0)] + pad[1:])
            dd = np.pad(dd, [(0, 0)] + pad[1:])
        for ax in (1, 2, 3):
            if rng.random() < 0.5:
                x, y, dd = np.flip(x, ax), np.flip(y, ax), np.flip(dd, ax)
        if self.jitter:
            for c in range(min(4, x.shape[0])):       # MRI channels only
                if rng.random() < 0.5:
                    m = x[c] != 0
                    x[c] = np.where(m, x[c] * rng.uniform(0.9, 1.1) + rng.uniform(-0.1, 0.1), 0.0)
        return (torch.from_numpy(np.ascontiguousarray(x)),
                torch.from_numpy(np.ascontiguousarray(y)),
                torch.from_numpy(np.ascontiguousarray(dd)))


# ===========================================================================
# E. losses / metrics / inference
# ===========================================================================
def distance_weighted_bce(logits, y, dist, tau=3.0, peak=5.0):
    w = 1.0 + peak * torch.exp(-dist.abs() / tau)
    return F.binary_cross_entropy_with_logits(logits, y, weight=w.expand_as(y), reduction="mean")


def _perimeter_weights(shape, alpha, device):
    fs = [torch.fft.fftfreq(shape[0], device=device, dtype=torch.float32) * 2 * math.pi,
          torch.fft.fftfreq(shape[1], device=device, dtype=torch.float32) * 2 * math.pi,
          torch.fft.rfftfreq(shape[2], device=device, dtype=torch.float32) * 2 * math.pi]
    g = torch.meshgrid(*fs, indexing="ij")
    k = torch.sqrt(sum(x ** 2 for x in g))
    w = (k / (k.max() + 1e-9)) ** alpha
    return w / (w.mean() + 1e-12)


class FractPerimLoss(nn.Module):
    """|k|^alpha seminorm of the probability map, normalised by the target's own
    seminorm -> one lambda is comparable across alpha."""

    def __init__(self, alpha=1.5):
        super().__init__()
        self.alpha = alpha

    def _semi(self, p):
        w = _perimeter_weights(p.shape[-3:], self.alpha, p.device)
        q = p.float() - p.float().mean(dim=(-3, -2, -1), keepdim=True)
        P = torch.fft.rfftn(q, dim=(-3, -2, -1)).abs() ** 2
        return (P * w).sum(dim=(-3, -2, -1)).mean() / float(np.prod(p.shape[-3:]))

    def forward(self, prob, target):
        with torch.no_grad():
            L0 = self._semi(target)
        return self._semi(prob) / (L0 + 1e-8)


class SegLoss(nn.Module):
    def __init__(self, boundary=True, frac_alpha=None, lam_frac=0.0):
        super().__init__()
        self.dice = DiceLoss(sigmoid=True, squared_pred=True, smooth_nr=0.0, smooth_dr=1e-5, batch=True)
        self.boundary = boundary
        self.frac = FractPerimLoss(frac_alpha) if (frac_alpha and lam_frac > 0) else None
        self.lam_frac = lam_frac

    def forward(self, logits, y, dist=None):
        loss = self.dice(logits, y)
        loss = loss + (distance_weighted_bce(logits, y, dist) if (dist is not None and self.boundary)
                       else F.binary_cross_entropy_with_logits(logits, y))
        if self.frac is not None:
            loss = loss + self.lam_frac * self.frac(torch.sigmoid(logits), y)
        return loss


def dice_np(p, t):
    ps, ts = int(p.sum()), int(t.sum())
    return 1.0 if ps + ts == 0 else 2.0 * int((p & t).sum()) / (ps + ts)


def hd95(pred, gt, penalty=373.13):
    from scipy.ndimage import binary_erosion, distance_transform_edt
    ps, gs = bool(pred.any()), bool(gt.any())
    if not ps and not gs:
        return 0.0
    if ps != gs:
        return penalty
    idx = np.where(pred | gt)
    sl = tuple(slice(max(0, int(i.min()) - 1), int(i.max()) + 2) for i in idx)
    p, g = pred[sl], gt[sl]
    sp, sg = p & ~binary_erosion(p), g & ~binary_erosion(g)
    d = np.concatenate([distance_transform_edt(~g)[sp], distance_transform_edt(~p)[sg]])
    return float(np.percentile(d, 95))


def band_mask(mask, r=3):
    from scipy.ndimage import binary_dilation, binary_erosion
    if not mask.any():
        return np.zeros_like(mask)
    return binary_dilation(mask, iterations=r) & ~binary_erosion(mask, iterations=r)


def band_dice(pred, gt, r=3):
    b = band_mask(gt, r)
    return float("nan") if not b.any() else dice_np(pred & b, gt & b)


def clean_components(mask, min_frac=0.1):
    from scipy.ndimage import label
    if not mask.any() or min_frac <= 0:
        return mask
    lab, n = label(mask)
    if n <= 1:
        return mask
    sizes = np.bincount(lab.ravel()); sizes[0] = 0
    keep = sizes >= min_frac * sizes.max(); keep[0] = False
    return keep[lab]


def postprocess(pred, thr=None, cc_wt=0.0, cc_tc=0.0, et_min=0):
    """THE single place where post-processing happens (val, test and tuning all call this)."""
    out = np.zeros_like(pred, bool)
    for c, cc in ((0, cc_wt), (1, cc_tc)):
        m = pred[c] if thr is None else pred[c]
        out[c] = clean_components(m, cc) if cc > 0 else m
    e = pred[2]
    out[2] = e if int(e.sum()) >= et_min else np.zeros_like(e)
    return out


@torch.no_grad()
def infer_probs(model, x_np, patch, overlap, device, amp=True, sw_batch=2, check=True,
                tag=""):
    """Sliding-window probabilities, (3, D, H, W) float16.

    `check=True` refuses to hand back a non-finite probability map.  This matters more than it
    looks: `p >= threshold` with `p = nan` is FALSE everywhere, so a poisoned checkpoint produces
    an EMPTY prediction, which the BraTS convention scores as 0.0 -- or as 1.0 when the reference
    for that region is also empty.  A nan model would therefore be certified and reported as a
    merely weak one, in silence.  Better to stop and name the checkpoint.

    Note that nothing like the training-time "skipped step" can happen here: inference has no
    backward pass, so there is no gradient to be non-finite and no weight to poison.  A non-finite
    probability map can only come from weights that were already nan (an old, pre-scaler
    checkpoint) or from an overflow in the input volumes.
    """
    try:
        from monai.inferers import sliding_window_inference
    except ImportError as _e:                        # pragma: no cover
        raise ImportError("MONAI (monai.inferers) is missing -> !pip install monai, then re-run.") from _e
    model.eval()
    xt = torch.from_numpy(x_np)[None].to(device)
    with torch.amp.autocast("cuda", enabled=amp, dtype=torch.float16):
        o = sliding_window_inference(xt, patch, sw_batch_size=sw_batch, predictor=model,
                                     overlap=overlap, mode="gaussian")
    if check:
        bad_x = int((~torch.isfinite(xt)).sum())
        bad_o = int((~torch.isfinite(o)).sum())
        if bad_o or bad_x:
            _w = max([float(q.abs().max()) for q in model.parameters()], default=0.0)
            raise RuntimeError(
                f"non-finite probabilities: {bad_o} of {o.numel()} logits are nan/inf "
                f"(input volumes: {bad_x} bad voxels; largest weight {_w:.3g})"
                f"{(' for ' + tag) if tag else ''}.\n"
                f"  Inference itself cannot overflow into nan -- there is no backward pass, so "
                f"there is no bad gradient to skip.  This means the WEIGHTS are already nan: the "
                f"checkpoint was written by a run that had already been poisoned (a build without "
                f"the GradScaler).  Do not evaluate or certify it: an all-nan map compares as an "
                f"EMPTY prediction, which the BraTS convention scores 0, or 1 when the reference "
                f"region is empty -- i.e. a dead model would look merely weak.  N1 detects this on "
                f"resume (it prints 'last.pt holds nan weights' and retrains that arm), and "
                f"code/check_run.py flags a run whose loss went non-finite; re-run N1 with "
                f"EXTEND=True (or delete that arm's *.pt) and it will be retrained.")
    return torch.sigmoid(o.float())[0].cpu().numpy().astype(np.float16)



# Bump this whenever the TTA maths changes: N2 folds it into the probability-cache provenance,
# so a cache written by an older implementation is recomputed instead of silently reused.
TTA_IMPL = 2


def infer_probs_tta(model, x_np, patch, overlap, device, flips=(), amp=True, sw_batch=2,
                    tag=""):
    """Flip-averaged sliding-window probabilities: the same model, the same weights, averaged
    over axis flips of the input.

    Why it is free accuracy: a segmentation network is not equivariant to flips, so each flipped
    pass makes a slightly different mistake; averaging the PROBABILITIES (before thresholding,
    where averaging is meaningful) removes part of that variance.  No training, no new weights,
    and the same treatment is applied to every arm, so the arm-to-arm comparison stays fair.

    flips : sequence of axis tuples, e.g. `[(), (0,), (1,), (2,)]` -- the identity plus one flip
            per spatial axis.  Each additional entry costs one more sliding-window pass.
            `()` or `None` means plain inference (no averaging, identical to `infer_probs`).
    """
    combos = [tuple(f) for f in (flips or [()])]
    if len(combos) <= 1:
        return infer_probs(model, x_np, patch, overlap, device, amp=amp, sw_batch=sw_batch,
                           check=True, tag=tag)
    acc = None
    for f in combos:
        # contiguous: np.flip produces negative strides, which torch's from_numpy refuses
        x = (np.ascontiguousarray(np.flip(x_np, axis=tuple(a + 1 for a in f)))
             if f else x_np)                                     # +1: channel axis first
        pr = infer_probs(model, x, patch, overlap, device, amp=amp, sw_batch=sw_batch,
                         check=True, tag=f"{tag} [flip {f or 'none'}]").astype(np.float32)
        if f:
            # `pr` is channel-first (3, D, H, W), exactly like the input -- so undoing a flip
            # along spatial axis `a` means axis `a + 1`.  Flipping `axis=f` here would flip the
            # CHANNEL axis for f=(0,), i.e. swap the WT/TC/ET maps and average three mutually
            # inconsistent volumes; that is a silent collapse of every Dice (observed: 0.84 ->
            # 0.03) rather than an error, which is why it needs a unit test, not a code review.
            pr = np.flip(pr, axis=tuple(a + 1 for a in f))
        acc = pr if acc is None else acc + pr
    out = (acc / len(combos)).astype(np.float16)
    if not np.isfinite(out).all():
        raise RuntimeError(f"flip-averaged probabilities are non-finite{(' for ' + tag) if tag else ''}")
    return out


# ===========================================================================
# F. kaggle plumbing
# ===========================================================================
class SessionBudget:
    """Graceful wall-clock guard: the notebook always exits cleanly and saves its
    checkpoints before the 9 h GPU cap hits."""

    def __init__(self, hours):
        self.t0 = time.time()
        self.limit = hours * 3600

    def left(self):
        return self.limit - (time.time() - self.t0)

    def ok(self, need=0.0):
        return self.left() > need

    def hhmm(self):
        s = max(0, int(self.left()))
        return f"{s//3600}h{(s%3600)//60:02d}m"


def run_done(run_dir):
    return os.path.exists(os.path.join(run_dir, "DONE"))


def mark_done(run_dir, payload=None):
    os.makedirs(run_dir, exist_ok=True)
    if payload is not None:
        json.dump(payload, open(os.path.join(run_dir, "DONE"), "w"), indent=2)
    else:
        open(os.path.join(run_dir, "DONE"), "w").write("ok")


def adopt_previous_runs(src, dst):
    """Copy everything from an attached dataset (previous sessions' output) into the
    working dir so the run queue can skip finished runs.  This is what makes the
    study resumable across Kaggle sessions, which do not persist /kaggle/working."""
    if not src or not os.path.isdir(src):
        print(f"adopt: nothing at {src}")
        return
    n = 0
    for root, _, files in os.walk(src):
        rel = os.path.relpath(root, src)
        for f in files:
            d = os.path.join(dst, rel, f)
            if not os.path.exists(d):
                os.makedirs(os.path.dirname(d), exist_ok=True)
                shutil.copy2(os.path.join(root, f), d)
                n += 1
    print(f"adopt: {n} new file(s) copied from {src} -> {dst}")


def disk_report(*paths):
    for p in paths:
        if os.path.isdir(p):
            tot = sum(os.path.getsize(os.path.join(r, f))
                      for r, _, fs in os.walk(p) for f in fs) / 1e9
            print(f"  {p:55s} {tot:6.2f} GB")
        elif os.path.exists(p):
            print(f"  {p:55s} {os.path.getsize(p)/1e6:6.1f} MB")
        else:
            print(f"  {p:55s} (missing)")


def peak_gpu_mem_gb():
    return torch.cuda.max_memory_allocated() / 1e9 if torch.cuda.is_available() else 0.0

"""safepp.py -- "safe post-processing": the mathematical core of the glioma reading aid.

Standalone (numpy + scipy only, no torch, no GPU, no Kaggle).  Everything here is
deterministic, unit-testable, and calibrated on VALIDATION data only.

Contents
--------
A. morphology / post-processing operators
     components()               26-connected component labelling
     filter_components()        remove components below a physical size
     relabel_et_to_necrosis()   nnU-Net BraTS post-processing step
     postproc_seg()             the two operators as one validated pipeline

B. risk control  (Angelopoulos et al., Conformal Risk Control, ICLR 2024)
     miss_curve()               per-case "fraction of tumour missed" vs threshold
     crc_threshold()            lambda-hat with the finite-sample correction
     stratified_crc()           group-conditional calibration per stratum

C. certified component filtering
     removal_curve()            fraction of true lesion volume a filter would delete
     certified_filter_threshold()

D. metrics
     dice_regions(), hd95(), lesionwise_dice(), component_stats(), volume_error()

Every public function has a docstring stating what it guarantees and what it does NOT
guarantee -- read those before quoting a number in a presentation.

Run `python3 safepp.py` for the self-test suite (no arguments, no data needed).
"""

import numpy as np
from scipy import ndimage

# --------------------------------------------------------------------------------------
# A. post-processing operators
# --------------------------------------------------------------------------------------

CONN26 = np.ones((3, 3, 3), dtype=bool)          # 26-connectivity for 3D components

# BraTS label convention used throughout the harness
NCR, ED, ET = 1, 2, 4
REGIONS = {"WT": (1, 2, 4), "TC": (1, 4), "ET": (4,)}


def region_thresholds(lam, params=None):
    """The threshold actually applied to each region's channel.

    WT -> the certified lambda, untouched.  TC/ET -> an optional per-region override from the
    frozen post-processing (tuned on validation like everything else).  The certificate is
    about WT, and the WT mask here is `seg > 0`, which depends only on the WT threshold -- so
    a TC/ET override cannot invalidate it.  That separation is the whole point: one shared
    threshold has to compromise between a large diffuse region and a small one, which is how
    cases end up with an empty ET prediction and a Dice of exactly 0.
    """
    lam = float(lam)
    p = params or {}
    out = {"WT": lam}
    for key, cap in (("thr_tc", "TC"), ("thr_et", "ET")):
        v = p.get(key)
        out[cap] = lam if v is None else float(v)
    return out


def labels_from_probs(p_wt, p_tc, p_et, thr):
    """Nested BraTS labels from three foreground probability maps.

    `thr` is either one number for all three regions or a dict/tuple of per-region thresholds
    (see `region_thresholds`).  Nested by construction: ET sits inside TC sits inside WT, so
    the label map can never contain an ET island outside the tumour.  This is the ONLY place
    where probabilities become a mask -- val, calibration and test all call it, which is what
    makes the certified threshold meaningful.
    """
    if isinstance(thr, dict):
        t_wt = float(thr.get("WT", 0.5))
        t_tc = float(thr.get("TC", t_wt))
        t_et = float(thr.get("ET", t_tc))
    elif isinstance(thr, (tuple, list, np.ndarray)):
        vals = [float(v) for v in thr]
        t_wt = vals[0]
        t_tc = vals[1] if len(vals) > 1 else t_wt
        t_et = vals[2] if len(vals) > 2 else t_tc
    else:
        t_wt = t_tc = t_et = float(thr)
    wt, tc, et = p_wt >= t_wt, p_tc >= t_tc, p_et >= t_et
    seg = np.zeros(p_wt.shape, np.uint8)
    seg[wt] = ED
    seg[wt & tc] = NCR
    seg[wt & tc & et] = ET
    return seg


def components(mask, conn=CONN26):
    """Label 26-connected components of a binary mask.

    Returns (labels int32, n_components).  `labels == 0` is background.
    """
    lab, n = ndimage.label(np.asarray(mask, dtype=bool), structure=conn)
    return lab.astype(np.int32), int(n)


def component_sizes(mask, conn=CONN26):
    """Sorted-descending array of component sizes (voxels).  Empty if mask is empty."""
    lab, n = components(mask, conn)
    if n == 0:
        return np.zeros(0, dtype=np.int64)
    return np.bincount(lab.ravel())[1:].astype(np.int64)


def filter_components(mask, min_voxels, conn=CONN26):
    """Keep only components with >= `min_voxels` voxels.

    Deterministic, idempotent, and monotone: raising the threshold can only remove more
    voxels.  That monotonicity is what makes the certified variant in section C valid.
    """
    mask = np.asarray(mask, dtype=bool)
    if min_voxels <= 1:
        return mask.copy()
    lab, n = components(mask, conn)
    if n == 0:
        return mask.copy()
    keep = np.bincount(lab.ravel(), minlength=n + 1) >= min_voxels
    keep[0] = False
    return keep[lab]


def relabel_et_to_necrosis(seg, min_et_voxels, ncr=NCR, et=ET):
    """nnU-Net's BraTS post-processing: if the enhancing tumour is smaller than
    `min_et_voxels`, relabel it to non-enhancing tumour/necrosis.

    Rationale (Isensee et al., BraTS 2020): tiny ET predictions are usually false
    positives near vessels/choroid plexus, and the challenge metric scores an empty
    prediction in an ET-less case as perfect.  Their paper also warns this *removes*
    true small enhancements, which is exactly why we certify it in section C.
    """
    seg = np.asarray(seg).copy()
    if min_et_voxels is None or min_et_voxels <= 0:
        return seg
    et_mask = seg == et
    if int(et_mask.sum()) < min_et_voxels:
        seg[et_mask] = ncr
    return seg


def postproc_seg(seg, params):
    """Apply the validated post-processing pipeline to a label map.

    params: dict(min_vox={region: int}, min_et_voxels=int, keep_largest=bool)
            `min_vox` is applied per region on the region mask; a voxel is removed only
            if it is not part of *any* kept region, so WT/TC/ET stay nested.
    """
    seg = np.asarray(seg).copy()
    min_vox = (params or {}).get("min_vox") or {}
    # 1. region-wise size filtering, nested: compute keep masks then re-derive labels
    wt_keep = filter_components(np.isin(seg, REGIONS["WT"]), int(min_vox.get("WT", 1)))
    tc_keep = filter_components(np.isin(seg, REGIONS["TC"]), int(min_vox.get("TC", 1)))
    et_keep = filter_components(np.isin(seg, REGIONS["ET"]), int(min_vox.get("ET", 1)))
    if (params or {}).get("keep_largest_wt"):
        lab, n = components(wt_keep)
        if n > 1:
            sizes = np.bincount(lab.ravel(), minlength=n + 1)
            wt_keep = lab == int(np.argmax(sizes[1:]) + 1)
    out = np.zeros_like(seg)
    out[wt_keep] = seg[wt_keep]                     # keep original labels inside WT
    out[~wt_keep] = 0
    et_removed = np.isin(seg, REGIONS["ET"]) & ~et_keep & wt_keep
    out[et_removed] = NCR                           # drop ET but keep it as core
    tc_removed = np.isin(seg, REGIONS["TC"]) & ~tc_keep
    out[tc_removed & (out != 0)] = ED               # drop TC -> falls back to edema
    # 2. ET relabeling last, so it sees the cleaned ET
    out = relabel_et_to_necrosis(out, (params or {}).get("min_et_voxels", 0))
    return out


# --------------------------------------------------------------------------------------
# B. conformal risk control
# --------------------------------------------------------------------------------------

def _as_prob(p):
    p = np.asarray(p, dtype=np.float32)
    return p


def miss_curve(prob, gt_fg, thresholds):
    """Per-threshold "fraction of ground-truth tumour voxels missed".

    prob    : (D,H,W) foreground probability
    gt_fg   : (D,H,W) boolean ground truth
    returns : array of shape (len(thresholds),) -- risk for "predict tumour if p >= t"

    Risk is non-decreasing in the threshold t (a stricter threshold recalls less), and
    bounded in [0, 1].  Empty ground truth -> risk 0 (nothing can be missed).
    """
    prob, gt_fg = _as_prob(prob), np.asarray(gt_fg, dtype=bool)
    n_fg = int(gt_fg.sum())
    if n_fg == 0:
        return np.zeros(len(thresholds), dtype=np.float64)
    v = prob[gt_fg]
    return np.array([float((v < t).mean()) for t in thresholds], dtype=np.float64)


def crc_threshold(calib_risks, alpha, B=1.0, thresholds=None):
    """Conformal risk control threshold with the finite-sample correction.

    calib_risks : (n_cal, n_thresholds) per-calibration-case risk, non-decreasing in the
                  threshold axis (as produced by `miss_curve`).
    alpha       : target risk level in (0, 1)
    B           : upper bound on the loss (1.0 for a fraction)

    Returns dict(threshold, lam, feasible, mean_risk_cal, bound_cal, risk_curve,
                 thresholds) where the guarantee is

        E[ risk(new case, threshold) ] <= alpha      (expectation over the calibration draw)

    Why the correction: with a finite calibration set the plain empirical quantile is
    optimistic; CRC adds B/(n+1) and rescales by n/(n+1), which yields the bound above
    with no distributional assumptions (Angelopoulos et al. 2024, Theorem 1).

    What it does NOT give: per-case coverage, or coverage inside a subgroup -- that is
    `stratified_crc`.  If no threshold satisfies the bound the function returns the most
    conservative threshold it has (t = min grid, i.e. the largest prediction set) with
    feasible=False: report that the model cannot meet this risk level, do not pretend.
    """
    R = np.asarray(calib_risks, dtype=np.float64)
    n = R.shape[0]
    if n == 0:
        raise ValueError("empty calibration set")
    if thresholds is None:
        thresholds = np.linspace(0, 1, R.shape[1])
    thresholds = np.asarray(thresholds, dtype=np.float64)
    assert R.shape[1] == len(thresholds), \
        f"risk array has {R.shape[1]} columns but {len(thresholds)} thresholds were given"
    mean_risk = R.mean(axis=0)
    bound = (n / (n + 1.0)) * mean_risk + B / (n + 1.0)
    ok = np.where(bound <= alpha)[0]
    if len(ok):
        # Tightest set that still meets the target: risk is non-decreasing in the
        # threshold, so the LARGEST feasible threshold is the smallest prediction set
        # that keeps the guarantee.  Taking ok[0] instead would return "predict
        # everything", which is a vacuous certificate.
        j = int(ok[-1])
        feasible = True
    else:
        j, feasible = 0, False            # no threshold meets alpha -> say so, don't fake it
    return dict(threshold=float(thresholds[j]), lam=float(1.0 - thresholds[j]),
                feasible=feasible, mean_risk_cal=float(mean_risk[j]),
                bound_cal=float(bound[j]), risk_curve=mean_risk, bound_curve=bound,
                thresholds=thresholds, n_cal=n)


def stratified_crc(calib_risks_by_group, alpha, B=1.0, thresholds=None):
    """Group-conditional CRC: one threshold per stratum.

    Group-conditional calibration gives E[risk | group] <= alpha for each group, instead
    of only E[risk] <= alpha overall.  The statistical price is a smaller calibration set
    per group (so a looser, larger prediction set) -- report that price, it is the honest
    part of the claim.

    returns {group: crc_threshold(...)} and, for convenience, the pooled threshold under
    key "__pooled__".
    """
    out = stratified_crc.__wrapped__ if False else {}
    pooled = np.concatenate([np.asarray(v) for v in calib_risks_by_group.values()], axis=0)
    out["__pooled__"] = crc_threshold(pooled, alpha, B=B, thresholds=thresholds)
    for g, R in calib_risks_by_group.items():
        out[g] = crc_threshold(np.asarray(R), alpha, B=B, thresholds=thresholds)
    return out


def fp_curve(prob, gt_fg, thresholds):
    r"""Fraction of the PREDICTION that is wrong: |pred_theta \ GT| / |pred_theta|.

    This is the over-segmentation counterpart of `miss_curve`: it FALLS as the threshold
    rises (a stricter threshold throws away the least confident -- usually spurious --
    voxels first).  Empty prediction -> 0 by convention (nothing predicted, nothing wrong).
    """
    prob, gt_fg = _as_prob(prob), np.asarray(gt_fg, dtype=bool)
    out = []
    for t in thresholds:
        pred = prob >= t
        n = int(pred.sum())
        out.append(0.0 if n == 0 else float((pred & ~gt_fg).sum()) / n)
    return np.array(out, dtype=np.float64)


def monotone_envelope(r, direction="increasing"):
    """Replace a noisy risk curve by its worst-case monotone envelope.

    Certifying the envelope certifies the deployed operator, because at every threshold
    the envelope is >= the realised risk:
      * increasing family -> running MAXIMUM over smaller thresholds
      * decreasing family -> running MAXIMUM over larger thresholds
    The alternative (`minimum.accumulate`) looks tighter and is WRONG: it would certify a
    risk lower than the operator actually carries.
    """
    r = np.asarray(r, dtype=np.float64)
    # ALONG THE THRESHOLD AXIS ONLY.  Calling np.maximum.accumulate without `axis` on a
    # (n_cal, n_thresholds) matrix flattens it in C order, so the running maximum leaks
    # from one calibration case into the next and the bounds inflate by a large factor
    # (observed: mean risk 0.055 -> envelope 0.34), which makes every certificate
    # infeasible and the deployed threshold needlessly conservative.
    if direction == "increasing":
        return np.maximum.accumulate(r, axis=-1)
    return np.maximum.accumulate(r[..., ::-1], axis=-1)[..., ::-1]


def certified_threshold_pair(calib_fn, calib_fp, thresholds, alpha, beta, B=1.0):
    """Two one-sided certificates -> one admissible threshold band.

    calib_fn : (n_cal, K) per-case MISSED-tumour fraction, increasing in the threshold
    calib_fp : (n_cal, K) per-case WRONG-prediction fraction, decreasing in the threshold
    alpha    : bound on the missed fraction      (under-segmentation risk)
    beta     : bound on the wrong-prediction fraction (over-segmentation risk)

    Rule (pre-registered, decided before looking at test): take the tightest threshold
    that meets the missed-tumour target, and accept it only if it also meets the
    wrong-prediction target.  If the two targets cannot be met together at this calibration
    size, `feasible` is False and you report that instead of a number.
    """
    F = monotone_envelope(np.asarray(calib_fn), "increasing")
    G = monotone_envelope(np.asarray(calib_fp), "decreasing")
    assert F.shape == G.shape, "risk arrays must have the same shape"
    n = F.shape[0]
    thresholds = np.asarray(thresholds, dtype=np.float64)
    assert F.shape[1] == len(thresholds), "risk arrays and thresholds disagree"
    fn_bound = (n / (n + 1.0)) * F.mean(axis=0) + B / (n + 1.0)
    fp_bound = (n / (n + 1.0)) * G.mean(axis=0) + B / (n + 1.0)
    ok_fn = np.where(fn_bound <= alpha)[0]        # prefix: risk grows with the threshold
    ok_fp = np.where(fp_bound <= beta)[0]         # suffix: risk falls with the threshold
    j_fn = int(ok_fn[-1]) if len(ok_fn) else None
    j_fp = int(ok_fp[0]) if len(ok_fp) else None
    feasible = (j_fn is not None) and (j_fp is not None) and (j_fn >= j_fp)
    j = j_fn if j_fn is not None else 0
    return dict(lam=float(thresholds[j]), feasible=bool(feasible),
                thr_fn=float(thresholds[j_fn]) if j_fn is not None else None,
                thr_fp=float(thresholds[j_fp]) if j_fp is not None else None,
                fn_bound_curve=fn_bound, fp_bound_curve=fp_bound,
                fn_mean_curve=F.mean(axis=0), fp_mean_curve=G.mean(axis=0),
                thresholds=thresholds, alpha=float(alpha), beta=float(beta), n_cal=n)


def choose_certificate(cert, cnr=None, stratified=True):
    """Which certificate should be presented for ONE case?

    The pooled certificate is valid on average over the cohort; a stratified one is valid
    within its subgroup and is therefore the right thing to show when it exists, is feasible,
    and we know the case's contrast.  Falling back to the pooled certificate is silently safe:
    the caller never gets a number out of a stratum whose certificate was infeasible.

    cert : the per-arm entry of certificates.json
    cnr  : the case's contrast-to-noise (wt_cnr); None/NaN -> pooled
    Returns dict(scope, lam, n_cal, curves, pooled_lam, cut) with scope 'low'|'high'|'pooled'.
    """
    strata = (cert or {}).get("strata") or None
    pooled = dict(scope="pooled", lam=(cert or {}).get("lam"), n_cal=(cert or {}).get("n_cal"),
                  curves=(cert or {}).get("curves"), pooled_lam=(cert or {}).get("lam"),
                  cut=(strata or {}).get("cut"))
    if not (stratified and strata) or cnr is None or not np.isfinite(float(cnr)):
        return pooled
    low = float(cnr) <= float(strata["cut"])
    lam = strata.get("lam_low" if low else "lam_high")
    feas = bool(strata.get("feasible_low" if low else "feasible_high"))
    if not (feas and lam is not None):
        return pooled                                   # stratum not certifiable -> pooled
    cur = strata.get("curves_low" if low else "curves_high") or pooled["curves"]
    return dict(scope=("low" if low else "high"), lam=float(lam),
                n_cal=int(strata["n_low" if low else "n_high"]), curves=cur,
                pooled_lam=(cert or {}).get("lam"), cut=float(strata["cut"]))


def volume_interval(v_pred, alpha, beta):
    """Worst-case volume interval implied by the two certified risk levels.

    If a case's missed fraction is at most alpha and its wrong-prediction fraction at most
    beta, then
        true >= (1 - beta) * predicted      (at least that share of the prediction is real)
        true <= predicted / (1 - alpha)     (the missed tumour cannot exceed that share)
    so the interval is guaranteed *given* those per-case risk levels.

    IMPORTANT: the certificates bound the AVERAGE risk over cases, not each individual
    case, so this is the worst-case consequence for a case sitting at the certified level.
    Always report the empirical containment rate on held-out cases next to it.
    """
    hi = float(v_pred) / max(1e-9, (1.0 - float(alpha)))
    lo = float(v_pred) * (1.0 - float(beta))
    return (min(lo, hi), max(lo, hi))


def volume_at(prob, gt_fg, threshold, voxel_vol_mm3=1.0):
    """Predicted volume (mm^3) at a threshold, with the ground-truth volume alongside."""
    prob = _as_prob(prob)
    pred = prob >= threshold
    return (float(pred.sum()) * voxel_vol_mm3,
            float(np.asarray(gt_fg, bool).sum()) * voxel_vol_mm3)


def per_case_risk(prob, gt_fg, threshold):
    """Realised risk for one case at one threshold (use this to audit a certificate)."""
    return float(miss_curve(prob, gt_fg, [threshold])[0])


# --------------------------------------------------------------------------------------
# C. certified component filtering
# --------------------------------------------------------------------------------------

def removal_curve(pred_mask, gt_seg, thresholds, region="WT", voxel_vol_mm3=1.0):
    """Per-threshold fraction of ground-truth lesion volume that filtering would delete.

    For each threshold d (in voxels): apply `filter_components` to the predicted region
    mask and measure the ground-truth voxels that the FILTER deleted -- i.e.
    (prediction before AND NOT after) AND ground truth, divided by the ground-truth
    volume.  The result is the filter's collateral damage, per case, in [0, 1].

    Note the subtlety (it changes the clinical claim): voxels the model never predicted
    are NOT charged to the filter, otherwise model error would masquerade as filter risk.

    Non-decreasing in d (see `filter_components` monotonicity), which is what makes the
    CRC bound below valid.
    """
    pred = np.isin(pred_mask, REGIONS[region]) if pred_mask.dtype != bool else pred_mask
    gt = np.isin(gt_seg, REGIONS[region])
    n_gt = int(gt.sum())
    if n_gt == 0:
        return np.zeros(len(thresholds), dtype=np.float64)
    out = []
    for d in thresholds:
        kept = filter_components(pred, int(d))
        removed = pred & ~kept                       # what the filter took away
        out.append(float((removed & gt).sum()) / n_gt)
    return np.array(out, dtype=np.float64)


def certified_filter_threshold(calib_removal, alpha, B=1.0, thresholds=None,
                              component_size_floor=None):
    """Largest component-size threshold whose collateral damage is risk-controlled.

    calib_removal : (n_cal, n_thresholds) from `removal_curve`, non-decreasing in d.
    Guarantee: E[deleted true-lesion volume fraction] <= alpha for a new case.
    component_size_floor : optional cap (voxels) on the returned threshold.  Needed
        because the guarantee is EXPECTATIONAL: if the calibration set happens to
        contain no true lesion between 60 and 1600 voxels, the damage curve is flat
        there and the certified threshold can land just below the first big lesion --
        statistically valid, clinically unnerving.  Set the floor from the observed
        distribution of true component sizes on validation (e.g. its 25th percentile)
        and report both numbers: "certified delta = X, engineering floor = Y".

    Because `filter_components` is monotone, the family of operators is nested, so the
    same CRC correction applies (index the family by -d, or equivalently search for the
    LARGEST d that satisfies the bound -- done below).
    """
    R = np.asarray(calib_removal, dtype=np.float64)
    n = R.shape[0]
    if thresholds is None:
        thresholds = np.arange(R.shape[1])
    thresholds = np.asarray(thresholds, dtype=np.float64)
    assert R.shape[1] == len(thresholds), \
        f"removal array has {R.shape[1]} columns but {len(thresholds)} thresholds were given"
    mean_risk = R.mean(axis=0)
    bound = (n / (n + 1.0)) * mean_risk + B / (n + 1.0)
    ok = np.where(bound <= alpha)[0]
    if len(ok):
        j = int(ok[-1])                   # LARGEST threshold that is still safe
        feasible = True
    else:
        j, feasible = 0, False
    delta, capped = float(thresholds[j]), False
    if component_size_floor is not None and delta > float(component_size_floor):
        delta, capped = float(component_size_floor), True
    return dict(threshold=delta, feasible=feasible, capped=capped,
                certified_threshold=float(thresholds[j]),
                mean_risk_cal=float(mean_risk[j]), bound_cal=float(bound[j]),
                risk_curve=mean_risk, bound_curve=bound, thresholds=thresholds, n_cal=n)


# --------------------------------------------------------------------------------------
# D. metrics
# --------------------------------------------------------------------------------------

def dice(a, b, eps=1e-8):
    a, b = np.asarray(a, dtype=bool), np.asarray(b, dtype=bool)
    return float(2.0 * (a & b).sum() / (a.sum() + b.sum() + eps))


def dice_regions(pred_seg, gt_seg):
    """{WT, TC, ET} voxel-wise Dice.  Empty-prediction convention: if both are empty the
    Dice is defined as 1.0 (the BraTS convention used in the nnU-Net paper)."""
    out = {}
    for r, labs in REGIONS.items():
        p, g = np.isin(pred_seg, labs), np.isin(gt_seg, labs)
        if p.sum() == 0 and g.sum() == 0:
            out[r] = 1.0
        else:
            out[r] = dice(p, g)
    return out


def _surf_dist(a, b, spacing=(1.0, 1.0, 1.0)):
    """Symmetric surface distances: for each surface voxel of a, distance to b (EDT)."""
    if a.sum() == 0 or b.sum() == 0:
        return None
    dt_b = ndimage.distance_transform_edt(~b, sampling=spacing)
    dt_a = ndimage.distance_transform_edt(~a, sampling=spacing)
    sa = a & ~ndimage.binary_erosion(a)          # 6-connectivity surface
    sb = b & ~ndimage.binary_erosion(b)
    if sa.sum() == 0 or sb.sum() == 0:
        return None
    d_ab, d_ba = dt_b[sa], dt_a[sb]
    return np.concatenate([d_ab, d_ba])


def hd95(pred_seg, gt_seg, region="WT", spacing=(1.0, 1.0, 1.0)):
    """95th percentile of the symmetric surface distance, in mm (NaN if undefined).

    Convention: if exactly one side is empty the HD95 is NaN -- do not silently score it
    as 0 or as a huge number; the caller decides (report the count of such cases).
    """
    p, g = np.isin(pred_seg, REGIONS[region]), np.isin(gt_seg, REGIONS[region])
    d = _surf_dist(p, g, spacing)
    return float("nan") if d is None else float(np.percentile(d, 95))


def lesionwise_dice(pred_seg, gt_seg, region="WT", min_gt_voxels=1, bbox_dilate=0,
                     gt_labels=None):
    """Lesion-wise Dice, averaged over ground-truth connected components of `region`.

    Convention (stated because implementations differ): the ground truth is partitioned
    into 26-connected components; for each component the prediction is restricted to that
    component's bounding box (optionally dilated by `bbox_dilate` voxels); a component
    with no predicted voxels at all scores 0.  Components smaller than `min_gt_voxels`
    are ignored.  Returns (mean, per_lesion_array); NaN if the ground truth has none.
    """
    g = np.isin(gt_seg, REGIONS[region])
    p = np.isin(pred_seg, REGIONS[region])
    # `gt_labels=(lab, n)` from `components(g)` lets a parameter sweep label the ground truth
    # ONCE instead of once per candidate: that is the difference between a few seconds and a
    # few minutes when a tuning grid calls this thousands of times, and it is the same answer.
    lab, n = gt_labels if gt_labels is not None else components(g)
    if n == 0:
        return float("nan"), np.zeros(0)
    scores = []
    for i in range(1, n + 1):
        comp = lab == i
        if comp.sum() < min_gt_voxels:
            continue
        sl = ndimage.find_objects(comp.astype(np.int32))[0]
        sl = tuple(slice(max(0, s.start - bbox_dilate), s.stop + bbox_dilate) for s in sl)
        pc = p[sl]
        scores.append(dice(pc.any() and (pc & comp[sl]), comp[sl]) if pc.any() else 0.0)
    scores = np.array(scores, dtype=np.float64)
    return (float(scores.mean()) if len(scores) else float("nan")), scores


def component_stats(pred_seg, gt_seg, region="WT"):
    """Component bookkeeping for the fragmentation story.

    returns dict(n_pred, n_gt, beta0_error=|n_pred-n_gt|, false_components, missed_lesions,
                 pred_voxels, gt_voxels)
    false_components : predicted components containing no ground-truth voxels
    missed_lesions   : ground-truth components containing no predicted voxels
    """
    p, g = np.isin(pred_seg, REGIONS[region]), np.isin(gt_seg, REGIONS[region])
    lp, npc = components(p)
    lg, ngc = components(g)
    false_c = 0
    for i in range(1, npc + 1):
        if not (g & (lp == i)).any():
            false_c += 1
    missed = 0
    for i in range(1, ngc + 1):
        if not (p & (lg == i)).any():
            missed += 1
    return dict(n_pred=npc, n_gt=ngc, beta0_error=abs(npc - ngc), false_components=false_c,
                missed_lesions=missed, pred_voxels=int(p.sum()), gt_voxels=int(g.sum()))


def volume_error(pred_seg, gt_seg, region="WT", voxel_vol_mm3=1.0):
    """(predicted - true) volume in mm^3 for one region of one case."""
    p = int(np.isin(pred_seg, REGIONS[region]).sum())
    g = int(np.isin(gt_seg, REGIONS[region]).sum())
    return float((p - g) * voxel_vol_mm3)


def bland_altman(errors):
    """Mean error and 95 % limits of agreement for an array of volume errors."""
    e = np.asarray(errors, dtype=np.float64)
    e = e[~np.isnan(e)]
    if len(e) == 0:
        return dict(n=0, bias=float("nan"), sd=float("nan"), lo=float("nan"), hi=float("nan"))
    bias, sd = float(e.mean()), float(e.std(ddof=1)) if len(e) > 1 else 0.0
    return dict(n=len(e), bias=bias, sd=sd, lo=bias - 1.96 * sd, hi=bias + 1.96 * sd)


# --------------------------------------------------------------------------------------
# self-tests -- no data, no GPU; run this file directly
# --------------------------------------------------------------------------------------

def _toy_case(seed=0, n=48):
    """A synthetic case: one big blobby lesion + optional small satellites."""
    rng = np.random.default_rng(seed)
    zz, yy, xx = np.mgrid[0:n, 0:n, 0:n].astype(np.float32)
    c = (n - 1) / 2.0
    big = (((xx - c) / 9) ** 2 + ((yy - c) / 7) ** 2 + ((zz - c) / 6) ** 2) <= 1
    small = (((xx - c - 15) / 2.5) ** 2 + ((yy - c + 12) / 2.5) ** 2 + ((zz - c) / 2.5) ** 2) <= 1
    tiny = (((xx - c + 17) / 1.5) ** 2 + ((yy - c - 14) / 1.5) ** 2 + ((zz - c + 8) / 1.5) ** 2) <= 1
    return big, small, tiny


def _test_postproc_operators():
    big, small, tiny = _toy_case()
    assert component_sizes(small)[0] > component_sizes(tiny)[0]
    # 1. filter removes exactly the small components, keeps the big one
    pred = (big | small | tiny)
    kept = filter_components(pred, int(component_sizes(small)[0]) + 1)
    assert np.array_equal(kept, big), "filter must keep only components >= threshold"
    # 2. monotonicity: raising the threshold never adds voxels
    for t in range(1, 400, 37):
        assert (filter_components(pred, t + 10) & ~filter_components(pred, t)).sum() == 0
    # 3. nested-region post-processing: the WT region must survive untouched, and a
    #    removed ET component must fall back to necrosis (never to a spurious label)
    seg = np.zeros_like(big, dtype=np.uint8)
    seg[big] = ED
    seg[small] = ET
    seg[tiny] = ET
    out = postproc_seg(seg, dict(min_vox={"WT": 1, "TC": 1,
                                          "ET": int(component_sizes(tiny)[0]) + 1},
                                 min_et_voxels=0))
    assert set(np.unique(out).tolist()) <= {0, NCR, ED, ET}, np.unique(out)
    assert np.isin(out, REGIONS["WT"]).sum() == np.isin(seg, REGIONS["WT"]).sum(), \
        "WT must be preserved when its own threshold is 1"
    assert (out[tiny] == NCR).all(), "the removed ET component must become necrosis"
    # 4. ET relabeling
    seg2 = np.zeros_like(big, dtype=np.uint8)
    seg2[big] = ED
    seg2[tiny] = ET
    out2 = relabel_et_to_necrosis(seg2, min_et_voxels=int(tiny.sum()) + 1)
    assert (out2 == ET).sum() == 0 and (out2 == NCR).sum() == int(tiny.sum()), "ET must become NCR"
    out3 = relabel_et_to_necrosis(seg2, min_et_voxels=1)
    assert (out3 == ET).sum() == int(tiny.sum()), "a large-enough ET must be kept"
    # 5. empty mask is handled everywhere
    z = np.zeros((8, 8, 8), bool)
    assert filter_components(z, 5).sum() == 0 and component_sizes(z).size == 0
    print("  [ok] post-processing operators (filter, monotonicity, nesting, ET relabel, empty)")


def _toy_probs(seed, n=28, quality=0.85):
    """GT = big|small; predicted probability = quality inside GT, small noise outside."""
    rng = np.random.default_rng(1000 + seed)
    big, small, _ = _toy_case(seed, n)
    gt = big | small
    p = rng.uniform(0.0, 0.35, size=(n, n, n)).astype(np.float32)
    p[gt] = np.clip(quality + rng.normal(0, 0.18, gt.sum()), 0, 1).astype(np.float32)
    return p, gt


def _test_crc_guarantee(n_splits=200, alpha=0.10, n_cal=37, n_test=37, pool_n=300):
    """The headline statistical property: averaged over calibration draws, the realised
    risk of the chosen threshold must not exceed alpha, and the certificate must be
    TIGHT (not the vacuous 'predict everything' solution)."""
    THR = np.linspace(0.0, 1.0, 201)
    pool = [_toy_probs(s) for s in range(pool_n)]
    rng = np.random.default_rng(0)
    realised, thresholds = [], []
    for k in range(n_splits):
        idx = rng.permutation(pool_n)
        cal = [pool[i] for i in idx[:n_cal]]
        tst = [pool[i] for i in idx[n_cal:n_cal + n_test]]
        R = np.stack([miss_curve(p, g, THR) for p, g in cal])
        d = crc_threshold(R, alpha=alpha, thresholds=THR)
        assert d["feasible"], "synthetic model should easily meet alpha=0.10"
        assert d["threshold"] > 0.05, "certificate must not degenerate to predict-everything"
        realised.append(np.mean([per_case_risk(p, g, d["threshold"]) for p, g in tst]))
        thresholds.append(d["threshold"])
    m = float(np.mean(realised))
    assert m <= alpha * 1.02, f"CRC violated: mean realised risk {m:.4f} > {alpha}"
    assert m >= alpha * 0.3, f"suspiciously conservative ({m:.4f}) -- check the formula"
    thr_sd = float(np.std(thresholds))
    print(f"       thresholds chosen: mean {np.mean(thresholds):.3f}, sd {thr_sd:.3f} "
          f"(calibration noise at n_cal={n_cal})")
    # alpha must actually matter: a tighter target yields a smaller realised risk
    R = np.stack([miss_curve(p, g, THR) for p, g in pool[:n_cal]])
    t_lo = crc_threshold(R, 0.05, thresholds=THR)["threshold"]
    t_hi = crc_threshold(R, 0.30, thresholds=THR)["threshold"]
    assert t_lo < t_hi, "smaller alpha must give a larger prediction set (lower threshold)"
    print(f"  [ok] CRC guarantee: mean realised risk {m:.4f} <= alpha {alpha} over "
          f"{n_splits} calibration draws (n_cal={n_cal}); alpha monotone")


def _test_envelope_axis():
    """Regression test for the axis bug described in `monotone_envelope`."""
    rows = np.array([[0.10, 0.20, 0.30],
                     [0.30, 0.10, 0.20]])          # second row has a dip
    up = monotone_envelope(rows, "increasing")
    assert np.allclose(up[0], [0.10, 0.20, 0.30]), up
    assert np.allclose(up[1], [0.30, 0.30, 0.30]), up
    assert up.max() <= 0.30 + 1e-12, "the envelope must not leak across calibration cases"
    down = monotone_envelope(rows, "decreasing")
    assert np.allclose(down[0], [0.30, 0.30, 0.30]) and np.allclose(down[1], [0.30, 0.20, 0.20])
    one_d = monotone_envelope(np.array([0.5, 0.2, 0.4]))
    assert np.allclose(one_d, [0.5, 0.5, 0.5]), one_d
    print("  [ok] monotone envelope acts on the threshold axis only (no cross-case leakage)")


def _test_volume_interval():
    """The two-sided certificate: both risks must hold on held-out data, and the implied
    volume interval must contain the truth for most cases."""
    THR = np.linspace(0.0, 1.0, 201)
    pool = [_toy_probs(s, quality=0.75) for s in range(200)]
    rng = np.random.default_rng(7)
    idx = rng.permutation(200)
    cal_i, tst_i = idx[:60], idx[60:160]
    FN = np.stack([monotone_envelope(miss_curve(pool[i][0], pool[i][1], THR)) for i in cal_i])
    FP = np.stack([monotone_envelope(fp_curve(pool[i][0], pool[i][1], THR), "decreasing")
                   for i in cal_i])
    for alpha, beta in ((0.05, 0.20), (0.10, 0.20), (0.15, 0.30)):
        d = certified_threshold_pair(FN, FP, THR, alpha=alpha, beta=beta)
        assert d["feasible"], f"targets {alpha}/{beta} should be reachable on this toy data"
        assert d["thr_fn"] >= d["thr_fp"], "the admissible band must be a real interval"
        r_fn, r_fp, inside = [], [], []
        for i in tst_i:
            p, g = pool[i]
            pred = p >= d["lam"]
            r_fn.append(float((g & ~pred).sum()) / max(1, int(g.sum())))
            r_fp.append(0.0 if pred.sum() == 0 else float((pred & ~g).sum()) / int(pred.sum()))
            lo, hi = volume_interval(float(pred.sum()), alpha, beta)
            inside.append(lo <= float(g.sum()) <= hi)
        m_fn, m_fp = float(np.mean(r_fn)), float(np.mean(r_fp))
        assert m_fn <= alpha + 0.03, f"missed-tumour certificate violated: {m_fn:.3f} > {alpha}"
        assert m_fp <= beta + 0.05, f"wrong-prediction certificate violated: {m_fp:.3f} > {beta}"
        cov = float(np.mean(inside))
        assert cov >= 0.75, f"volume interval contains the truth only {cov:.0%} of the time"
        print(f"       alpha={alpha:.2f} beta={beta:.2f} -> lambda*={d['lam']:.2f} "
              f"(band [{d['thr_fp']:.2f},{d['thr_fn']:.2f}]), realised FN {m_fn:.3f}, "
              f"FP {m_fp:.3f}, interval containment {cov:.0%}")
    # the two targets move the band in OPPOSITE directions -- that is the whole point:
    #   * a tighter missed-tumour target needs an EARLIER (lower) threshold -> bigger set
    #   * a tighter wrong-prediction target raises the FLOOR of the band
    d_a_lo = certified_threshold_pair(FN, FP, THR, alpha=0.05, beta=0.20)
    d_a_hi = certified_threshold_pair(FN, FP, THR, alpha=0.30, beta=0.20)
    assert d_a_lo["thr_fn"] <= d_a_hi["thr_fn"], (d_a_lo["thr_fn"], d_a_hi["thr_fn"])
    d_b_lo = certified_threshold_pair(FN, FP, THR, alpha=0.20, beta=0.05)
    d_b_hi = certified_threshold_pair(FN, FP, THR, alpha=0.20, beta=0.40)
    assert d_b_lo["thr_fp"] >= d_b_hi["thr_fp"], (d_b_lo["thr_fp"], d_b_hi["thr_fp"])
    print(f"       monotone: alpha 0.05->0.30 moves thr_fn {d_a_lo['thr_fn']:.2f}->"
          f"{d_a_hi['thr_fn']:.2f}; beta 0.05->0.40 moves thr_fp {d_b_lo['thr_fp']:.2f}->"
          f"{d_b_hi['thr_fp']:.2f}")
    print("  [ok] two-sided certificates + volume interval (both risks hold on held-out cases)")


def _test_stratified():
    """Low-quality cases must get their own (more conservative) threshold."""
    THR = np.linspace(0, 1, 201)
    easy = [_toy_probs(s, quality=0.95) for s in range(30)]
    hard = [_toy_probs(s + 500, quality=0.60) for s in range(30)]
    R_easy = np.stack([miss_curve(p, g, THR) for p, g in easy])
    R_hard = np.stack([miss_curve(p, g, THR) for p, g in hard])
    res = stratified_crc({"low_contrast": R_hard, "high_contrast": R_easy}, alpha=0.10,
                         thresholds=THR)
    assert res["low_contrast"]["threshold"] < res["high_contrast"]["threshold"], \
        "the harder stratum must receive the lower (more inclusive) threshold"
    pooled = res["__pooled__"]["threshold"]
    assert res["low_contrast"]["threshold"] <= pooled, "pooling hides the hard stratum"
    print(f"  [ok] stratified CRC: hard stratum threshold {res['low_contrast']['threshold']:.2f} "
          f"< easy {res['high_contrast']['threshold']:.2f} (pooled {pooled:.2f})")


def _test_certified_filter():
    """A filter that deletes true satellites must be bounded by alpha, and the bound must
    stop it from ever deleting the essential (large) lesion."""
    big_size = int(_toy_case(0)[0].sum())
    THR = np.array([1, 10, 25, 50, 100, 200, 400, 800, 1600, 2400])
    cal, tst = [], []
    for s in range(60):
        big, small, tiny = _toy_case(s)
        gt = (big | small | tiny).astype(np.uint8)
        pred = (big | small | tiny).astype(np.uint8)
        if s % 3 == 0:                        # the model sometimes misses the satellite
            pred[small] = 0
        (cal if s < 40 else tst).append(removal_curve(pred, gt, THR, region="WT"))
    d = certified_filter_threshold(np.stack(cal), alpha=0.10, thresholds=THR)
    assert d["feasible"], "a modest filter must be certifiable at alpha=0.10"
    j = int(np.where(THR == d["threshold"])[0][0])
    realised = float(np.mean([r[j] for r in tst]))
    # 1. the guarantee holds on held-out cases
    assert realised <= 0.10 + 1e-9, realised
    # 2. the certificate protects essential anatomy: it must never pick a threshold that
    #    would delete the big lesion
    assert d["threshold"] < big_size, (d["threshold"], big_size)
    # 3. a naive fixed threshold is NOT certified -> the certificate added something
    RC = np.stack(cal)
    jbad = int(np.where(THR == 2400)[0][0])          # 2400 > the big lesion's size
    bound_bad = (RC.shape[0] / (RC.shape[0] + 1.0)) * RC[:, jbad].mean() + 1.0 / (RC.shape[0] + 1.0)
    assert bound_bad > 0.10, f"a 2400-voxel filter must fail the bound (got {bound_bad:.2f})"
    # 4. damage is charged to the filter, not to the model: a perfectly missed satellite
    #    must not appear as filter damage
    gt_only = (big | small | tiny).astype(np.uint8)
    missed = np.zeros_like(gt_only); missed[big] = 2
    r = removal_curve(missed, gt_only, np.array([50.0]), region="WT")
    assert r[0] == 0.0, "voxels the model never predicted must not count as filter damage"
    # 5. the engineering floor caps a statistically valid but unnerving threshold
    d_cap = certified_filter_threshold(np.stack(cal), alpha=0.10, thresholds=THR,
                                       component_size_floor=200.0)
    assert d_cap["capped"] and d_cap["threshold"] == 200.0 and d_cap["certified_threshold"] == 1600.0, d_cap
    assert d_cap["threshold"] < big_size
    _c = dict(lam=0.42, n_cal=37, curves=dict(fn=[0.3], fp=[0.1]),
              strata=dict(cut=11.4, n_low=12, n_high=25, lam_low=0.34, lam_high=0.46,
                          feasible_low=True, feasible_high=True,
                          curves_low=dict(fn=[0.2], fp=[0.08]),
                          curves_high=dict(fn=[0.1], fp=[0.05])))
    assert choose_certificate(_c, cnr=9.0)["lam"] == 0.34
    assert choose_certificate(_c, cnr=9.0)["scope"] == "low"
    assert choose_certificate(_c, cnr=15.0)["lam"] == 0.46
    assert choose_certificate(_c, cnr=15.0)["n_cal"] == 25
    assert choose_certificate(_c, cnr=np.nan)["scope"] == "pooled"
    assert choose_certificate(_c, cnr=9.0, stratified=False)["lam"] == 0.42
    _c["strata"]["feasible_low"] = False           # never hand back a dead stratum
    assert choose_certificate(_c, cnr=9.0)["scope"] == "pooled"
    assert choose_certificate(dict(lam=0.5, n_cal=10), cnr=9.0)["scope"] == "pooled"
    print("[ok] choose_certificate: stratum when certifiable, pooled otherwise")

    print(f"       floor: certified {int(d_cap['certified_threshold'])} capped to "
          f"{int(d_cap['threshold'])} by the observed lesion-size floor")
    print(f"  [ok] certified filter: delta={int(d['threshold'])} voxels (< big lesion "
          f"{big_size}), cal bound {d['bound_cal']:.3f}, realised collateral {realised:.3f} "
          f"<= 0.10; a naive delta=2400 is rejected (bound {bound_bad:.2f})")


def _test_metrics():
    big, small, tiny = _toy_case()
    gt = np.zeros(big.shape, np.uint8)
    gt[big] = ED
    gt[small] = ET
    gt[tiny] = ET
    pred = np.zeros_like(gt)
    pred[big] = ED
    pred[small] = ET                      # tiny is missed entirely
    # lesion-wise dice: ET has 2 GT components, one predicted, one missed -> mean in (0,1)
    m, per = lesionwise_dice(pred, gt, region="ET")
    assert len(per) == 2 and 0.0 <= m <= 1.0 and per.min() == 0.0, (m, per)
    st = component_stats(pred, gt, region="ET")
    assert st["missed_lesions"] == 1 and st["false_components"] == 0, st
    # a spurious island must show up as a false component (that is the whole point of the
    # fragmentation metric), while a missed lesion must be counted separately
    pred2 = pred.copy()
    pred2[2:5, 2:5, 2:5] = ET             # spurious blob in the corner, no GT there
    st2 = component_stats(pred2, gt, region="ET")
    assert st2["false_components"] == 1 and st2["missed_lesions"] == 1, st2
    assert st2["n_pred"] == 2 and st2["n_gt"] == 2 and st2["beta0_error"] == 0, st2
    # WT: three disconnected GT blobs, two of them predicted
    st3 = component_stats(pred, gt, region="WT")
    assert st3["n_gt"] == 3 and st3["n_pred"] == 2 and st3["missed_lesions"] == 1, st3
    assert st3["beta0_error"] == 1, st3
    d = dice_regions(pred, gt)
    assert d["ET"] < 1.0 and d["WT"] > 0.5, d
    # HD95 is NaN when one side is empty, and finite otherwise
    assert not np.isnan(hd95(pred, pred, "WT")), "identical masks must give HD95 = 0"
    empty = np.zeros_like(pred)
    assert np.isnan(hd95(empty, gt, "ET")), "undefined HD95 must be NaN, not 0"
    ve = volume_error(pred, gt, "WT")
    assert ve < 0, "this prediction misses a lesion, so the volume error must be negative"
    ba = bland_altman([ve, ve])
    assert ba["n"] == 2 and ba["sd"] == 0.0 and ba["bias"] == ve, ba
    print("  [ok] metrics: lesion-wise Dice, component stats, Dice, HD95 conventions, Bland-Altman")


def _selftest():
    print("safepp self-test")
    _test_postproc_operators()
    _test_metrics()
    _test_envelope_axis()
    _test_crc_guarantee()
    _test_volume_interval()
    _test_stratified()
    _test_certified_filter()
    print("all self-tests passed")


# (self-tests disabled inside the notebook; run code/safepp.py standalone)

# ============================================================================
# F. v2 additions: the structure-recall target and its patch sampler
# ============================================================================
class SkelPatchDataset(PatchDataset):
    """PatchDataset + the thin ground-truth target of the three regions.

    N0 precomputes {pid}_skel.npz (WT/TC/ET thin targets, see skeletonize_mask and
    component_recall_loss) once; training then never has to thin anything on the GPU.  Sample
    i is chosen EXACTLY as in PatchDataset (same seed, same foreground centre, same flips), so
    the target patch is voxel-aligned with the label patch -- `selftest_skel_dataset()`
    verifies that alignment (kept deliberately strict).
    """

    def __init__(self, cache, ids, length, skel_root=None, **kw):
        super().__init__(cache, ids, "base", length, **kw)   # arm is unused here
        self.skel_root = as_roots(skel_root) or self.cache

    def __getitem__(self, i):
        rng = np.random.default_rng([self.seed, self.offset + i])
        pid = self.ids[int(rng.integers(len(self.ids)))]
        mri = np.load(self._p(pid, "mri"), mmap_mode="r")
        seg = np.load(self._p(pid, "seg"), mmap_mode="r")
        dist = np.load(self._p(pid, "dist"), mmap_mode="r")
        shape = seg.shape

        center = None
        if rng.random() < self.P_FG:
            locs = np.load(self._p(pid, "locs", "npz"))
            present = [k for k in ("c1", "c2", "c4") if len(locs[k]) > 0]
            if present:
                k = present[int(rng.integers(len(present)))]
                center = locs[k][int(rng.integers(len(locs[k])))]
        if center is None:
            center = [rng.integers(s) for s in shape]
        st = [int(np.clip(c - p // 2, 0, max(0, s - p))) for c, s, p in zip(center, shape, self.patch)]
        sl = tuple(slice(a, a + p) for a, p in zip(st, self.patch))

        x = np.asarray(mri[(slice(None),) + sl], dtype=np.float32)
        y = seg_to_targets(np.asarray(seg[sl]))
        dd = decode_dist(np.asarray(dist[sl]))[None]
        f = _first_file(self.skel_root, f"{pid}_skel.npz")
        with np.load(f) as z:
            sk = np.stack([np.asarray(z[k][sl], np.float32) for k in ("wt", "tc", "et")])

        pad = [(0, 0)] + [(0, p - s) for p, s in zip(self.patch, x.shape[1:])]
        if any(b for _, b in pad):
            x = np.pad(x, pad); y = np.pad(y, [(0, 0)] + pad[1:])
            dd = np.pad(dd, [(0, 0)] + pad[1:]); sk = np.pad(sk, [(0, 0)] + pad[1:])
        for ax in (1, 2, 3):
            if rng.random() < 0.5:
                x, y, dd, sk = (np.flip(a, ax) for a in (x, y, dd, sk))
                break
        # (a single flip pass, exactly as PatchDataset does; the generator above keeps the
        #  RNG stream identical to the parent because all four arrays are flipped together)
        for c in range(min(4, x.shape[0])):
            if rng.random() < 0.5:
                m = x[c] != 0
                x[c] = np.where(m, x[c] * rng.uniform(0.9, 1.1) + rng.uniform(-0.1, 0.1), 0.0)
        return (torch.from_numpy(np.ascontiguousarray(x)),
                torch.from_numpy(np.ascontiguousarray(y)),
                torch.from_numpy(np.ascontiguousarray(dd)),
                torch.from_numpy(np.ascontiguousarray(sk)))


def component_recall_loss(prob, target, eps=1e-6):
    """Soft recall of the PREDICTION on a thin GROUND-TRUTH target (the lesion's core).

        L = 1 - sum(p * T) / sum(T)

    `target` is `centre_ridge(mask)`: the Euclidean-distance ridge of the mask, computed per
    26-connected component (see N0 / `build_skel_*`).  What that buys, in one sentence per
    case:

      * elongated structure (enhancing rim, vessel-like ET, a thin edema arm) -> the ridge is
        a CURVE along its middle, so the term demands that the prediction cover the whole
        structure, not just its fat end;
      * a compact lesion -> the ridge is a short core near its centre;
      * every ground-truth COMPONENT gets at least one ridge voxel, so the term directly
        penalises MISSING A SATELLITE LESION -- a beta_0 (component-count) recall, which is
        the failure mode this project actually has.

    HONEST SCOPE: this is NOT the published Skeleton Recall Loss (ECCV 2024, arXiv
    2404.03010) and must not be quoted as it.  The published method thins the GT with Lee's
    algorithm into a homotopy-equivalent skeleton; the ridge used here is thinner-and-centred
    but not homotopy-preserving.  The FORM is identical (one overlap term, no differentiable
    skeletonisation, a few tensor ops), and `SKEL_BACKEND='skimage'` swaps the target for
    Lee's thinning when that build passes `_skimage_skeleton_ok()` -- i.e. you can run the
    published variant from the same code, and N0 prints which target it built.

    `prob` and `target` are (B, C, D, H, W); the reduction is over voxels and classes.
    """
    # the sums ask for fp32 explicitly: `torch.sum` on a half tensor returns a half, and a
    # whole-patch sum of fp16 values overflows 65504 -> inf -> nan (exactly the Dice problem)
    num = (prob * target).sum(dtype=torch.float32)
    den = target.sum(dtype=torch.float32) + eps
    return 1.0 - num / den


# old name kept so nothing that already imports it breaks
skeleton_recall_loss = component_recall_loss


class DiceLossFast(nn.Module):
    """MONAI `DiceLoss(sigmoid=True, squared_pred=True, smooth_nr=0.0, smooth_dr=1e-5, batch=True)`
    reimplemented so the ELEMENTWISE maths stays in the autocast dtype (fp16 on GPU = fast) while
    every REDUCTION accumulates in fp32.  Verified against MONAI in fp32 (same value to ~1e-5,
    `code/test_dice_fast.py`).

    Why this exists -- it is not a micro-optimisation, it is the bug that killed two GPU runs:
    the reduction in MONAI's DiceLoss is `torch.sum(half_tensor)`, which ACCUMULATES in fp32 but
    RETURNS A HALF.  Over a 96^3 patch that result is ~1.8e5, i.e. above fp16's 65504 -> inf ->
    nan loss -> nan weights -> a model that predicts nothing for the whole run (`loss nan` for
    8000 iterations, `best=0.0556`, in both arms).  Casting the whole loss to fp32 fixes it but
    makes the elementwise ops (and their backward) fp32 as well, measured ~5x slower per
    iteration on the user's T4 (0.33 -> 1.6 s/it).  Here the sums ask for fp32 explicitly:
    `(p * p).sum(dtype=torch.float32)`, so nothing overflows and nothing is wasted.
    """

    def __init__(self, smooth_nr=0.0, smooth_dr=1e-5, batch=True):
        super().__init__()
        self.smooth_nr, self.smooth_dr, self.batch = float(smooth_nr), float(smooth_dr), bool(batch)

    def forward(self, logits, targets):
        p = torch.sigmoid(logits)                       # autocast dtype: fp16 on GPU
        y = targets if targets.dtype == p.dtype else targets.to(p.dtype)
        dims = list(range(2, p.ndim)) + ([0] if self.batch else [])
        # MONAI's order and formula, so the value matches to rounding
        intersection = (p * y).sum(dim=dims, dtype=torch.float32)
        ground_o = (y * y).sum(dim=dims, dtype=torch.float32)
        pred_o = (p * p).sum(dim=dims, dtype=torch.float32)
        f = 1.0 - (2.0 * intersection + self.smooth_nr) / (ground_o + pred_o + self.smooth_dr)
        return f.mean()


class TopologyLoss(nn.Module):
    """dice+BCE plus an optional structure-recall term whose strength is SCHEDULED by the
    training loop (`self.w` is a plain float, ramped from 0 to the arm's nominal value).

    Why not a learnable multiplier -- measured, not assumed: with `w` as an nn.Parameter the
    gradient of the loss w.r.t. w is +R(theta) > 0 at whatever theta the model currently has,
    so plain gradient descent drives w below zero, relu(w) kills the term and the arm becomes
    silently identical to its control (observed: w = -0.006 after 25 iterations, i.e. off).
    A learnable weight can only earn its place through a meta-gradient over the whole training
    run, which a single scalar under SGD does not have.

    What is kept is the property that actually matters: w starts at ZERO, so step 0 and the
    whole warm-up phase are bit-identical to the Dice+BCE control, and whether the term earns
    its place is decided by the held-out metrics (N2/N3), not by the scalar itself.
    """

    def __init__(self, w0=0.0, boundary=True):
        super().__init__()
        self.dice = DiceLossFast()      # fp16-safe + fast; same value as MONAI's DiceLoss
        self.boundary = boundary
        self.w = float(w0)                       # ramped by the training loop; not a parameter
        self.nominal = float(w0)

    def forward(self, logits, y, dist=None, skel=None):
        loss = self.dice(logits, y)
        loss = loss + (distance_weighted_bce(logits, y, dist)
                       if (dist is not None and self.boundary)
                       else F.binary_cross_entropy_with_logits(logits, y))
        if skel is not None and float(self.w) > 0:
            loss = loss + float(self.w) * component_recall_loss(torch.sigmoid(logits), skel)
        return loss


def _skimage_skeleton_ok():
    """Is this build's skimage.skeletonize trustworthy?

    NOT a formality: builds exist that return an ALL-ZERO skeleton for shapes as simple as a
    solid cube (observed in the wild with scikit-image 0.25.x).  A silently empty skeleton
    would make the topology arm train on nothing, so we detect it and fall back.
    Returns (ok, note).
    """
    try:
        from skimage.morphology import skeletonize
    except Exception as e:                                   # pragma: no cover
        return False, f"skimage unavailable ({e})"
    zz, yy, xx = np.mgrid[0:32, 0:32, 0:32]
    probes = (("cube8", np.ones((8, 8, 8), bool), 4),
              ("tube4x4x17", _probe_tube(), 8),
              ("ellipsoid5x6x7", (((zz - 16) / 5.0) ** 2 + ((yy - 16) / 6.0) ** 2
                                  + ((xx - 16) / 7.0) ** 2) <= 1, 4))
    notes = []
    for name, m, floor in probes:
        try:
            n = int(np.asarray(skeletonize(m.astype(bool))).sum())
        except Exception as e:                               # pragma: no cover
            return False, f"skeletonize raised on {name} ({e})"
        if n < floor:
            return False, (f"skeletonize is degenerate here: {name} ({int(m.sum())} voxels "
                           f"in) -> {n} skeleton voxel(s), expected >= {floor}")
        notes.append(f"{name}={n}")
    return True, "ok (" + ", ".join(notes) + ")"


def _probe_tube():
    t = np.zeros((10, 10, 20), bool)
    t[3:7, 3:7, 2:19] = True
    return t


def centre_ridge(mask):
    """Dependency-free centre line: the ridge of the Euclidean distance transform.

    Voxels that sit inside the mask and are a local maximum of the distance to the background
    in their 26-neighbourhood.  For a tube that is a curve, for a blob a medial sheet -- both
    are thin and centred, which is what the skeleton-recall term needs (it asks how much of
    the centre of the lesion the prediction covers).

    Guarantee: the result is a subset of the mask and is never empty when the mask is not.
    It is NOT a homotopy-equivalent thinning; `skeletonize_mask` prefers Lee thinning when
    that is available and trustworthy.
    """
    from scipy import ndimage
    m = np.asarray(mask, bool)
    if not m.any():
        return m.copy()
    # pad by one background voxel: without it the distance transform of a mask that touches
    # the array border measures the distance to the array corner instead of to the boundary
    mp = np.pad(m, 1)
    d = ndimage.distance_transform_edt(mp)
    ridge = mp & (d >= ndimage.maximum_filter(d, size=3))     # local maxima of the distance
    if not ridge.any():                                       # single-voxel mask, etc.
        ridge = mp & (d == d.max())
    lab, n = ndimage.label(mp, structure=np.ones((3, 3, 3), int))
    for i in range(1, n + 1):                                 # every component represented
        comp = lab == i
        if not (ridge & comp).any():
            ridge |= comp & (d == d[comp].max())
    return ridge[1:-1, 1:-1, 1:-1]


def skeletonize_mask(mask, backend="auto", verbose=False):
    """Thin, centred skeleton of `mask`, with a self-checked backend.

    backend='auto'             skimage's 3-D Lee thinning if that build passes a sanity check,
                               otherwise the dependency-free distance-transform ridge
    backend='skimage'|'ridge'  force one
    """
    m = np.asarray(mask, bool)
    if not m.any():
        return m.copy()
    if backend == "auto":
        ok, note = _skimage_skeleton_ok()
        backend = "skimage" if ok else "ridge"
        if verbose:
            print(f"  skeleton backend: {backend} ({note})")
    if backend == "skimage":
        from skimage.morphology import skeletonize
        out = np.asarray(skeletonize(m.astype(bool))).astype(bool)
        if out.any():
            return out
    return centre_ridge(m)


def skeleton_selftest(verbose=True):
    """Fail loudly if the skeletons the topology arm will train on are degenerate."""
    zz, yy, xx = np.mgrid[0:24, 0:24, 0:24]
    blob = (((zz - 11) / 5.0) ** 2 + ((yy - 11) / 6.0) ** 2 + ((xx - 11) / 7.0) ** 2) <= 1
    tube = np.zeros((24, 24, 24), bool)
    tube[10:14, 10:14, 3:21] = True
    out = {}
    for name, m in (("blob", blob), ("tube", tube), ("blob+tube", blob | tube),
                    ("cube8", np.ones((8, 8, 8), bool))):
        sk = skeletonize_mask(m)
        assert sk.any(), f"{name}: empty skeleton for a non-empty mask"
        assert not (sk & ~m).any(), f"{name}: skeleton outside the mask"
        frac = sk.sum() / max(1, int(m.sum()))
        assert frac < 0.9, f"{name}: skeleton is not thinner than the mask ({frac:.2f})"
        out[name] = (int(m.sum()), int(sk.sum()))
    ok, note = _skimage_skeleton_ok()
    if verbose:
        print(f"[ok] skeletonize_mask {out} | skimage backend usable: {ok} ({note})")
    return out


def selftest_skel_dataset(cache, ids, patch=(32, 32, 32), n=8, seed=42, skel_root=None):
    """Verify the skeleton patch is voxel-aligned with the targets (skel[0] must be the WT
    skeleton, i.e. nonzero exactly where y[0] is nonzero for a non-empty target patch)."""
    a = PatchDataset(cache, ids, "base", length=n, patch=patch, seed=seed, dod_tag=None)
    b = SkelPatchDataset(cache, ids, length=n, patch=patch, seed=seed, skel_root=skel_root)
    for i in range(n):
        x1, y1, d1 = a[i]
        x2, y2, d2, s = b[i]
        assert torch.equal(y1, y2), f"targets differ at sample {i}"
        assert torch.equal(d1, d2), f"distance maps differ at sample {i}"
        assert s.shape == y1.shape, (s.shape, y1.shape)
        # WT skeleton must sit inside the WT target (allowing the 1-voxel skeleton margin)
        assert float((s[0] * (1 - y1[0])).sum()) <= 0.02 * float(y1[0].sum()) + 5, \
            f"skeleton not aligned with the target at sample {i}"
    return True


# ---- aliases: the cells below say bratsx.x() / safepp.x() -------------------
# In a notebook this cell runs in __main__, so both names are simply THIS
# namespace: nothing to import, nothing to attach, and a forked DataLoader
# worker resolves them for free.  (The fallback covers the case where the
# library is exec'd somewhere other than __main__, e.g. a test harness.)
import sys as _sys, types as _types
_ns = globals()
bratsx = _sys.modules.get(__name__)
if getattr(bratsx, "__dict__", None) is not _ns:
    bratsx = _types.ModuleType("bratsx")
    bratsx.__dict__.update(_ns)
safepp = bratsx
print("library ready:", len([k for k in dir(bratsx) if not k.startswith('_')]),
      "names (bratsx and safepp are aliases of this notebook's namespace)")


# %% [CELL 90] REPORT TEMPLATE (inlined GUI) -------------------------------
# The interactive report template, base64-inlined so this notebook needs nothing
# but its own inputs.  Written out by the last cell as a self-contained HTML file.
import base64 as _b64

_TPL_B64 = (
    "..."
)
REPORT_TEMPLATE = _b64.b64decode(_TPL_B64).decode("utf-8")
print(f"report template: {len(REPORT_TEMPLATE)} chars")
# %% [CELL 5] LOCATE INPUTS --------------------------------------------------
import base64, glob, io, json, os, time
import numpy as np
import pandas as pd
from scipy import ndimage


def _find_all(tag, probe):
    hits = [p for p in glob.glob(os.path.join(INPUT_ROOT, "**", "*"), recursive=True)
            if os.path.isdir(p) and os.path.basename(p).startswith(tag)
            and glob.glob(os.path.join(p, probe))]
    return sorted(set(hits))


def _first(name, extra=()):
    """First existing copy of a file, preferring an attached Kaggle dataset over /kaggle/working."""
    hits = sorted(glob.glob(os.path.join(INPUT_ROOT, "**", name), recursive=True)) + list(extra)
    for h in hits:
        if os.path.exists(h):
            return h
    return None


CACHE_BASE = bratsx.as_roots(CACHE_BASE) or _find_all("cache_base", "*_mri.npy") or \
    bratsx.as_roots(os.path.join(WORK, "cache_base"))
assert CACHE_BASE, "cache not found -- attach the N0 output dataset(s)"

if PROBS_DIR is None:
    PROBS_DIR = next(iter(_find_all("probs", os.path.join("*", "*.npz"))), None) or \
        os.path.join(WORK, "probs")
ARMDIR = PROBS_DIR if os.path.isdir(os.path.join(PROBS_DIR, "p_wt")) else None
if ARMDIR is None:                        # dataset root holds the arm folders directly
    ARMDIR = os.path.join(PROBS_DIR, ARM)
    if not os.path.isdir(ARMDIR):
        cand = sorted(glob.glob(os.path.join(PROBS_DIR, f"*{ARM}*")))
        ARMDIR = cand[0] if cand else ARMDIR
assert os.path.isdir(ARMDIR), f"no probability files for arm '{ARM}' under {PROBS_DIR}"

CERT_F = _first("certificates.json", [os.path.join(WORK, "certificates.json")])
POST_F = _first("frozen_postproc.json", [os.path.join(WORK, "frozen_postproc.json")])
assert CERT_F and POST_F, "certificates.json / frozen_postproc.json missing -- run N2 first"
CERT = json.load(open(CERT_F))
POST = json.load(open(POST_F))
SPLIT_F = _first("dataset_split.json", [os.path.join(WORK, "dataset_split.json")])
SPLIT_JSON = json.load(open(SPLIT_F)) if SPLIT_F else {}
SUM_F = _first("summary.json", [os.path.join(WORK, "summary.json")])
SUMMARY = json.load(open(SUM_F)) if SUM_F else {}
COV_F = _first("coverage_table.csv", [os.path.join(WORK, "test_csvs", "coverage_table.csv"),
                                            os.path.join(WORK, "coverage_table.csv")])
COVERAGE = pd.read_csv(COV_F) if COV_F else None
T_F = _first(f"{ARM}_test.csv", [os.path.join(WORK, "test_csvs", f"{ARM}_test.csv")])
TEST_DF = pd.read_csv(T_F) if T_F else None
STRATA_F = _first("strata.csv", [os.path.join(WORK, "strata.csv")])
STRATA = pd.read_csv(STRATA_F).set_index("pid") if STRATA_F else None
_mfs = sorted(glob.glob(os.path.join(INPUT_ROOT, "**", "manifest_base*.csv"), recursive=True)) or \
    sorted(glob.glob(os.path.join(WORK, "manifest_base*.csv")))
MAN = pd.concat([pd.read_csv(f) for f in _mfs]).drop_duplicates("pid").set_index("pid") \
    if _mfs else pd.DataFrame()

_ARM_SETTING = ARM    # [merge 2026-10-09] the arm the probabilities / test table were located with
ARM = (next((a for a in (ARM_PREFERENCE or []) if a in CERT), None)
       or (ARM if ARM in CERT else next(iter(CERT))))
# [merge 2026-10-09] ARMDIR and TEST_DF were located with the SETTINGS arm BEFORE ARM_PREFERENCE picked the
# delivered arm: the page could then show the ensemble's certificate on the base model's probabilities.
# Re-locate them for the arm actually reported.  ARM_STRICT (default True) stops if they are missing;
# set ARM_STRICT = False in the settings cell to downgrade that to a warning.
ARM_STRICT = globals().get("ARM_STRICT", True)
if ARM != _ARM_SETTING and not os.path.isdir(os.path.join(PROBS_DIR, "p_wt")):
    _cand = os.path.join(PROBS_DIR, ARM)
    if not os.path.isdir(_cand):
        _g = sorted(glob.glob(os.path.join(PROBS_DIR, f"*{ARM}*")))
        _cand = _g[0] if _g else _cand
    if os.path.isdir(_cand):
        ARMDIR = _cand
        T_F = _first(f"{ARM}_test.csv", [os.path.join(WORK, "test_csvs", f"{ARM}_test.csv")])
        TEST_DF = pd.read_csv(T_F) if T_F else TEST_DF
        print(f"  (re-located for the reported arm '{ARM}': {ARMDIR})")
    else:
        _msg = (f"probabilities for the delivered arm '{ARM}' are not under {PROBS_DIR}: the report would "
                f"show {ARM}'s certificate on '{_ARM_SETTING}' probabilities")
        assert not ARM_STRICT, _msg + "  (attach them, or set ARM_STRICT = False to accept this)"
        print("!! WARNING: " + _msg)
print(f"reporting arm: {ARM}  (available: {sorted(CERT)})")
CA = CERT[ARM]
A, B = float(CA["alpha"]), float(CA["beta"])
LAM = float(CA["lam"])
LAMBDAS = np.asarray(CA["curves"]["lams"], float)
REG = str(CA.get("region", PRIMARY)).upper()   # certificates.json wins if it disagrees
CUT = CA.get("strata", {}).get("cut", None)
print(f"arm {ARM} | lambda*={LAM:.2f} feasible={CA['feasible']} | alpha={A:.2f} beta={B:.2f} "
      f"| n_cal={CA['n_cal']} | region {REG}")
if not CA.get("valid", True):
    print(f"!! {CA.get('warning')}")
print(f"frozen post-processing: {POST}")
print(f"probabilities: {ARMDIR}")


# %% [CELL 6] RENDERING HELPERS ---------------------------------------------
def to_png(arr, mode="L", alpha=None):
    from PIL import Image
    a = np.asarray(arr)
    if mode == "RGBA":
        alpha = np.asarray(alpha, np.uint8)
        rgb = a.astype(np.uint8) if a.ndim == 3 else \
            np.repeat(a.astype(np.uint8)[..., None], 3, axis=2)
        a = np.dstack([rgb, alpha])
    buf = io.BytesIO()
    Image.fromarray(a.astype(np.uint8), mode).save(buf, "PNG", optimize=True)
    return "data:image/png;base64," + base64.b64encode(buf.getvalue()).decode()


def gray_slice(vol, z=None, lo=1, hi=99):
    vol = np.asarray(vol)
    s = (vol if z is None or vol.ndim == 2 else vol[..., z]).astype(np.float32)
    s = s.T                                    # (x, y) -> image row = y, col = x
    nz = s[np.abs(s) > 1e-6]
    if nz.size:
        p1, p99 = np.percentile(nz, [lo, hi])
        s = np.clip((s - p1) / (p99 - p1 + 1e-6), 0, 1)
    return (s * 255).astype(np.uint8)


def outline(m, t=1):
    m = np.asarray(m, bool)
    return m & ~ndimage.binary_erosion(m, iterations=t)


COLORS = {"gt": (63, 185, 80), "pred": (240, 136, 62), "post": (88, 166, 255)}
# Region colours, shared with the page: WT red, TC amber, ET cyan (same hex values as the
# template's RC map, so a screenshot and the legend cannot drift apart).
REGION_RGB = {"w": (229, 83, 75), "c": (210, 153, 34), "e": (57, 197, 207)}


def region_layer(m2d, key, fill=90, edge=150):
    """One region of one mask, in its own colour: fill + a 2-voxel bright edge."""
    m = np.asarray(m2d, bool).T
    rgb = np.zeros(m.shape + (3,), np.uint8)
    rgb[...] = REGION_RGB[key[-1]]
    al = (m * fill + outline(m, 2) * edge).astype(np.uint8)
    return to_png(rgb, "RGBA", al)


def mask_layer(m2d, which, fill=110, edge=150):
    m = np.asarray(m2d, bool).T
    rgb = np.zeros(m.shape + (3,), np.uint8)
    rgb[...] = COLORS[which]
    al = (m * fill + outline(m, 2) * edge).astype(np.uint8)
    return to_png(rgb, "RGBA", al)


def err_layer(e2d):
    e = np.asarray(e2d).T
    rgb = np.zeros(e.shape + (3,), np.uint8)
    rgb[e == 1] = (248, 81, 73)                # false positive
    rgb[e == 2] = (31, 111, 235)               # false negative
    return to_png(rgb, "RGBA", ((e > 0) * 235).astype(np.uint8))


def load_case(pid):
    z = np.load(os.path.join(ARMDIR, f"{pid}.npz"))
    return z, np.load(bratsx.cache_file(CACHE_BASE, pid, "seg")), \
        np.load(bratsx.cache_file(CACHE_BASE, pid, "mri"))


def pipeline(z, lam, params=None):
    # identical to N2's apply_pipeline: the per-region TC/ET thresholds frozen there apply here
    # too, so the mask drawn on the page is the mask N2 measured
    params = POST if params is None else params
    thr = safepp.region_thresholds(lam, params)
    raw = safepp.labels_from_probs(z["p_wt"], z["p_tc"], z["p_et"], thr)
    return raw, safepp.postproc_seg(raw, params)


def dice_wt(pred_seg, gt_seg):
    return float(safepp.dice_regions(pred_seg, gt_seg)[REG])


def cnr_of(pid):
    if STRATA is not None and pid in STRATA.index and "wt_cnr" in STRATA.columns:
        return float(STRATA.loc[pid, "wt_cnr"])
    return float(MAN.loc[pid, "wt_cnr"]) if (len(MAN) and "wt_cnr" in MAN.columns
                                             and pid in MAN.index) else float("nan")


def stratum_of(pid):
    if STRATA is not None and pid in STRATA.index and "stratum" in STRATA.columns:
        return str(STRATA.loc[pid, "stratum"])
    c = cnr_of(pid)
    if not np.isfinite(c) or not CA.get("strata"):
        return "unknown"
    return "low" if c <= float(CA["strata"]["cut"]) else "high"


def cert_for(pid, stratified=None):
    """Which certificate this case is reported against: its own stratum when that is
    certifiable, the pooled one otherwise (safepp.choose_certificate makes that call)."""
    return safepp.choose_certificate(CA, cnr_of(pid),
                                     stratified=STRATIFIED if stratified is None else stratified)


def lam_for(pid):
    return float(cert_for(pid)["lam"] or LAM)


def cert_note(pid, ch, c):
    """Say, in words, exactly which certificate is on screen and why -- including the two cases
    where a subgroup exists but must NOT be quoted."""
    sg = stratum_of(pid)
    if ch["scope"] != "pooled":
        return (f" This scan is <b>{ch['scope']}-contrast</b> (CNR {c:.1f}, cohort cut {CUT:.1f}) "
                f"and is reported against the certificate calibrated on <b>that subgroup alone</b> "
                f"(n_cal = {ch['n_cal']}, λ = {ch['lam']:.2f}), not the pooled one (λ = {LAM:.2f}). "
                f"The curves below are that subgroup's worst-case envelopes, so dragging α/β here "
                f"re-selects λ inside the subgroup.")
    out = f" The certificate shown is the <b>pooled</b> one (n_cal = {ch['n_cal']})."
    if sg in ("low", "high") and CUT is not None and np.isfinite(c):
        feas = (CA.get("strata") or {}).get("feasible_low" if sg == "low" else "feasible_high")
        if not feas:
            out += (f" This scan is <b>{sg}-contrast</b> (CNR {c:.1f} vs cut {CUT:.1f}), but that "
                    f"subgroup has no certificate at this α/β, so the pooled one is used rather "
                    f"than quoting a subgroup number that does not hold." + _low_evidence(sg))
        else:
            out += (f" It is <b>{sg}-contrast</b> (CNR {c:.1f} vs cut {CUT:.1f}); the subgroup was "
                    f"certifiable, but stratified reporting is switched off (STRATIFIED=False).")
    elif CUT is not None and np.isfinite(c):
        out += f" Its CNR is {c:.1f} (cohort cut {CUT:.1f})."
    return out


# %% [CELL 7] THE PAYLOAD THE GUI READS --------------------------------------
# ---------------------------------------------------------------------------
# Coarse volumes for the interactive parts of the report
#
# The 2D layers are pre-rendered PNGs AT THE CERTIFIED λ*, so on their own they cannot react to
# the α/β sliders -- which is exactly the "the sliders do nothing" complaint.  So the page also
# gets a coarse copy of the volumes it needs to re-decide things live:
#
#   * quantised probability volumes (WT/TC/ET) on a coarse grid -> the browser re-thresholds them
#     as you drag α/β, so the mask, the volume and the interval all move together;
#   * the ground-truth label volumes + a coarse MRI -> the 3D viewer, coloured by region.
#
# Encoding: one grayscale PNG per volume, the axial planes stacked vertically (height = ny*nz).
# The browser decodes it with the same <img>/canvas path it already uses -- no library, no
# network -- and a probability field compresses far better as PNG than as run-length coding
# (run-length is only good for the near-binary masks; on smooth fields it was ~4x bigger).
#
# Everything numeric (volumes, Dice, coverage) stays full resolution: the coarse volume is a
# VISUAL AID and the report says so.
def _vol_png(u8, n):
    """One data-URI PNG holding a whole volume: `n` axial planes concatenated top to bottom."""
    nz, ny, nx = (int(v) for v in n)
    from PIL import Image
    a = np.asarray(u8, np.uint8).reshape(nz, ny, nx)
    # pad to a whole number of planes so the browser can index it back without bookkeeping
    tall = np.concatenate([a[zi] for zi in range(nz)], axis=0) if nz else a.reshape(0, nx)
    out = io.BytesIO()
    Image.fromarray(tall, "L").save(out, "PNG", optimize=True)
    return "data:image/png;base64," + base64.b64encode(out.getvalue()).decode("ascii")

def _coarse_shape(shape, n_max):
    f = float(n_max) / float(max(shape))
    return tuple(max(1, int(round(s * f))) for s in shape)


def _pool3(a, target, how="mean"):
    """Downsample to `target` keeping the aspect ratio.  mean for probabilities (a coarse voxel
    is the average of what it covers), threshold-after-mean for masks (so a thin structure
    survives as a partial value rather than disappearing)."""
    a = np.asarray(a, np.float32)
    if tuple(a.shape) == tuple(target):
        out = a
    else:
        z = [t / float(s) for t, s in zip(target, a.shape)]
        out = ndimage.zoom(a, z, order=1, mode="nearest")
    return out


def vol3d_pack(pid, n_max=None):
    """The coarse bundles the 3D viewer and the live-lambda preview read.

    AXIS ORDER: every coarse volume is (a0, a1, a2) = (x, y, z), C-order, which is what the apps decode
    (verified against report_BraTS20_Training_095: n == full == (X, Y, Z)).
    [merge 2026-10-09] (1) the delivered mask was frozen at the POOLED lambda although meta.lam and every
    other layer use lam_for(pid); a low-contrast case therefore showed a delivered mask that did not belong
    to its own certificate.  (2) the anatomy volume was divided by its positive 99th percentile, but the
    cache is z-scored inside the brain, so about half the brain clipped to 0; it is now windowed by the
    brain's own 1-99 percentile.
    """
    n_max = int(n_max or VIEW3D_N)
    z, seg, mri = load_case(pid)
    shape = seg.shape
    tgt = _coarse_shape(shape, n_max)
    gtw = np.isin(seg, safepp.REGIONS["WT"]); gtc = np.isin(seg, safepp.REGIONS["TC"])
    gte = np.isin(seg, safepp.REGIONS["ET"])
    q = lambda p: np.rint(np.clip(_pool3(p, tgt, "mean"), 0, 1) * 255).astype(np.uint8)
    # thresholds are 0/255 in the PNG, NOT 0/1: the browser divides by 255
    mask = lambda m: (_pool3(m.astype(np.float32), tgt, "mean") >= 0.25).astype(np.uint8) * 255
    _, post = pipeline(z, float(lam_for(pid)))
    pw = np.isin(post, safepp.REGIONS["WT"]); pc = np.isin(post, safepp.REGIONS["TC"])
    pe = np.isin(post, safepp.REGIONS["ET"])
    m = _pool3(mri[0], tgt, "mean")
    inside = _pool3((np.abs(mri).max(axis=0) > 1e-6).astype(np.float32), tgt, "mean") >= 0.5
    if inside.any():
        lo, hi = np.percentile(m[inside], [1, 99])
        m = np.clip((m - lo) / (hi - lo + 1e-6), 0, 1) * inside
    planes = {}
    for k, arr in (("wt", q(z["p_wt"])), ("tc", q(z["p_tc"])), ("et", q(z["p_et"])),
                   ("gtw", mask(gtw)), ("gtc", mask(gtc)), ("gte", mask(gte)),
                   ("postw", mask(pw)), ("postc", mask(pc)), ("poste", mask(pe)),
                   ("mri", np.rint(m * 255).astype(np.uint8))):
        planes[k] = _vol_png(arr, tgt)
    return dict(n=list(map(int, tgt)), full=list(map(int, shape)), mri_max=255, planes=planes)


MESH_N = 96          # the COMMON frame the page works in (same call as vol3d_pack, so the meshes
                     # and the volume share one coordinate system and one physical scale per axis)
MESH_CELLS = 96      # cells across the longest side of each surface's own bounding box.  Because
                     # the field is blurred BEFORE resampling (MESH_BLUR_ORIG), this can be finer than
                     # the voxel grid without drawing the voxel grid -- a 30-voxel lesion gets 96
                     # cells of smooth surface.  Resampling a SHARP mask is the thing that terraces.
MESH_BRAIN_CELLS = 128
MESH_BLUR_ORIG = 1.2  # Gaussian sigma in ORIGINAL voxels, applied at full resolution BEFORE any
                      # resampling.  This is the load-bearing constant: it turns the mask into a
                      # smooth field, so no later step can draw the voxel grid, and it moves the
                      # visible boundary by ~1 mm (stated in the caption, never a measurement).
MESH_SIGMA = 0.7      # extra smoothing in CELLS after pooling, to round the level set a little more
MESH_MAX_FACES = 34000


def _area_pool(a, target, blur=MESH_BLUR_ORIG):
    """Blur-then-decimate: the mean of what a coarse voxel covers, and never an upsample.

    `_pool3` (used by the 2D volume views, where a soft image is what you want) is a LINEAR
    interpolation via `ndimage.zoom`.  On a binary mask that is a trap: the interpolated field
    crosses 0.5 exactly on the original voxel faces, so marching cubes rebuilds the voxel grid as
    terraces.  The fix is an ORDER: blur at full resolution first, resample second.  After that the
    field is a smooth function, and sampling it finely is legitimate -- a 96-cell grid over a
    30-voxel lesion is representing a smooth surface, not inventing resolution.  What must never
    happen is resampling a SHARP mask, in either direction.
    """
    a = np.asarray(a, np.float32)
    tgt = tuple(int(max(4, t)) for t in target)
    if blur:
        a = ndimage.gaussian_filter(a, blur)      # BEFORE any resampling -- this is the whole point
    if tuple(a.shape) == tgt:
        return a
    return ndimage.zoom(a, [t / float(s) for t, s in zip(tgt, a.shape)], order=1, mode="nearest")


def _mesh_one(field, shape, tgt, box, cells, level=0.5, sigma=MESH_SIGMA, step=1):
    """Marching cubes on a CROPPED grid, returned in the common `tgt` frame.

    `box` is (lo, hi) in common-frame index space (float, exclusive).  The crop is blurred, area-pooled
    to at most `cells` across its longest side, extracted, and mapped back into the common frame --
    so the page keeps one coordinate system and the geometry gets the resolution the data supports.
    """
    try:
        from skimage import measure
    except Exception:
        return None
    # two conversions, named so they cannot be swapped again:
    #   box is in COMMON-frame indices -> original crop needs  cpr = shape/tgt
    #   vertices are in ORIGINAL indices -> common frame needs  rpc = tgt/shape
    cpr = [float(s) / float(t) for s, t in zip(shape, tgt)]        # original voxels per common voxel
    rpc = [float(t) / float(s) for s, t in zip(shape, tgt)]        # common units per original voxel
    lo_o = [int(np.floor(b * c)) for b, c in zip(box[0], cpr)]
    hi_o = [int(np.ceil(b * c)) for b, c in zip(box[1], cpr)]
    for i in range(3):
        lo_o[i] = max(0, min(lo_o[i], shape[i] - 1)); hi_o[i] = max(lo_o[i] + 2, min(hi_o[i], shape[i]))
    crop = np.asarray(field, np.float32)[lo_o[0]:hi_o[0], lo_o[1]:hi_o[1], lo_o[2]:hi_o[2]]
    if crop.size == 0:
        return None
    ext = [float(h - l) for h, l in zip(hi_o, lo_o)]
    f = cells / max(1.0, max(ext))
    # `cells` across the longest side, up or down: after the pre-blur in _area_pool that is just a
    # sampling density, and a fine grid is what keeps the normals (and so the shading) smooth.
    n_loc = tuple(int(max(8, round(e * f))) for e in ext)
    m = _area_pool(crop, n_loc)
    if sigma:
        m = ndimage.gaussian_filter(m, sigma)
    if float(m.max()) < level:
        return None
    try:
        v, fa, _, _ = measure.marching_cubes(m, level=level, spacing=(1.0, 1.0, 1.0), step_size=step)
    except Exception:
        return None
    if fa.shape[0] > MESH_MAX_FACES and step == 1:
        return _mesh_one(field, shape, tgt, box, cells, level=level, sigma=sigma, step=2)
    for i in range(3):
        v[:, i] = (lo_o[i] + v[:, i] * ext[i] / n_loc[i]) * rpc[i]     # -> common-frame units
    return v.astype(np.float32), fa.astype(np.int64)


def jsonable(obj):
    """A payload with no NaN / Infinity in it.

    Python writes bare `NaN` for a missing value (an empty reference gives a NaN Dice, for example).
    That is valid in a JavaScript literal -- which is how the report page receives the payload -- but
    it is NOT valid JSON, so anything that parses the .json sidecar with a strict parser rejects the
    whole file.  The sidecar is written through this, so the same payload can be fetched by another
    front end (the reading-aid app) without a silent parse failure.  Non-finite numbers become null;
    a consumer that needs the old behaviour can treat null as "not available".
    """
    if isinstance(obj, dict):
        return {k: jsonable(v) for k, v in obj.items()}
    if isinstance(obj, (list, tuple)):
        return [jsonable(v) for v in obj]
    if isinstance(obj, (bool, str)) or obj is None:
        return obj
    if isinstance(obj, (int, np.integer)):
        return int(obj)
    if isinstance(obj, (float, np.floating)):
        return float(obj) if np.isfinite(obj) else None
    return obj


def _extent(box, shape, tgt):
    """Lengths of a common-frame box in ORIGINAL voxels (for logging and density decisions)."""
    cpr = [float(s) / float(t) for s, t in zip(shape, tgt)]
    return [(b - a) * c for a, b, c in zip(box[0], box[1], cpr)]


def _bbox_of(field, shape, tgt, level=0.5, pad_frac=0.06, pad_min=2.0):
    """(lo, hi) of `field > level` in common-frame index space, padded a little."""
    low = _pool3(np.asarray(field, np.float32), tgt, "mean")
    idx = np.nonzero(low > level * 0.8)
    if len(idx[0]) == 0:
        return None
    lo, hi = [], []
    for i in range(3):
        a, b = float(idx[i].min()), float(idx[i].max() + 1)
        pad = max(pad_min, pad_frac * (b - a))
        lo.append(max(0.0, a - pad)); hi.append(min(float(tgt[i]), b + pad))
    return (lo, hi)


def _mesh_pack(meshes, tgt):
    """Quantise to uint16 common-frame coordinates + uint16/uint32 indices, base64 them."""
    out = {}
    scale = 60000.0 / max(1.0, float(max(tgt) - 1))
    for k, m in meshes.items():
        if m is None:
            out[k] = None
            continue
        v, f = m
        nv, nf = int(v.shape[0]), int(f.shape[0])
        q = np.rint(np.clip(v, 0, None) * scale).astype(np.uint16)
        big = nv > 65535
        fi = f.astype(np.uint32) if big else f.astype(np.uint16)
        out[k] = dict(nv=nv, nf=nf, f32=bool(big), scale=float(scale),
                      v=_b64.b64encode(q.tobytes()).decode("ascii"),
                      f=_b64.b64encode(fi.tobytes()).decode("ascii"))
    return out


def mesh3d_pack(pid, v3=None):
    """Smooth surfaces for the 3D panel: prediction at the certified cut, the radiologist's regions,
    the delivered (post-processed) mask, and the head itself.

    Design notes that decide how this looks on screen:
      * every layer is extracted on ITS OWN cropped grid (`MESH_CELLS` across the lesion, so the
        surface has ~90 cells of resolution whatever the volume size), then mapped into the common
        `tgt` frame the volume uses -- the page never sees more than one coordinate system;
      * fields are mean-pooled and Gaussian-smoothed (~1.3 of the layer's own voxels) before
        extraction, which is what removes the voxel staircase.  Cost: the surface sits about one
        voxel outside the true mask edge, invisible at presentation size and stated here so nobody
        reads the render as a measurement;
      * the head is a low-level iso-surface of the (already skull-stripped) MRI, smoothed harder:
        it is context, drawn as a translucent shell behind the tumour.
    """
    v3 = v3 or vol3d_pack(pid)
    z, seg, mri = load_case(pid)
    shape = tuple(int(x) for x in seg.shape)
    tgt = _coarse_shape(shape, MESH_N)
    lam = float(lam_for(pid))
    pw = z["p_wt"] >= lam
    ptc = np.logical_and(z["p_tc"] >= lam, pw)
    pet = np.logical_and(z["p_et"] >= lam, ptc)
    _, post = pipeline(z, lam)
    fields = dict(pred_wt=pw, pred_tc=ptc, pred_et=pet,
                  gt_wt=np.isin(seg, safepp.REGIONS["WT"]),
                  gt_tc=np.isin(seg, safepp.REGIONS["TC"]),
                  gt_et=np.isin(seg, safepp.REGIONS["ET"]),
                  post_wt=np.isin(post, safepp.REGIONS["WT"]),
                  post_tc=np.isin(post, safepp.REGIONS["TC"]),
                  post_et=np.isin(post, safepp.REGIONS["ET"]))
    meshes = {}
    for k, m in fields.items():
        m = np.asarray(m)
        box = _bbox_of(m, shape, tgt, level=0.5)
        meshes[k] = None if box is None else _mesh_one(m.astype(np.float32), shape, tgt, box,
                                                       MESH_CELLS)
    anat = mri[0].astype(np.float32)
    nz_ = anat[anat > 0]
    if nz_.size:
        anat = np.clip(anat / (np.percentile(nz_, 99) + 1e-6), 0, 1)
    abox = _bbox_of(anat, shape, tgt, level=0.12, pad_frac=0.02, pad_min=1.0)
    # level 0.18 of the normalised MRI: high enough to sit on the brain outline (not on the
    # skull-strip edge), low enough that the shell encloses the whole volume -- it is the frame the
    # tumour has to be seen inside, so it must not shrink to a blob
    meshes["brain"] = None if abox is None else _mesh_one(anat, shape, tgt, abox,
                                                          MESH_BRAIN_CELLS, level=0.18, sigma=2.0)
    dens = []
    for k in ("gt_wt", "gt_tc", "gt_et"):
        bx = _bbox_of(np.asarray(fields[k]), shape, tgt, level=0.5)
        if bx is not None:
            dens.append(int(min(MESH_CELLS, max(_extent(bx, shape, tgt)))))
    if dens:
        print("      mesh density: %d..%d cell(s) across the reference surfaces (cap %d)"
              % (min(dens), max(dens), MESH_CELLS))
    return dict(n=list(map(int, tgt)), cells=int(MESH_CELLS), layers=_mesh_pack(meshes, tgt))


MESH_LOOK = {                     # key -> (label, colour, opacity) -- same palette as the page
    "brain":   ("head",                             "#aab4c6", 0.10),
    "gt_wt":   ("reference · WT",                   "#e5534b", 0.20),
    "gt_tc":   ("reference · TC",                   "#d29922", 0.34),
    "gt_et":   ("reference · ET",                   "#39c5cf", 0.52),
    "post_wt": ("delivered · WT",                    "#58a6ff", 0.16),
    "post_tc": ("delivered · TC",                    "#58a6ff", 0.30),
    "post_et": ("delivered · ET",                    "#58a6ff", 0.46),
    "pred_wt": ("prediction · WT (lambda*)",         "#e5534b", 0.26),
    "pred_tc": ("prediction · TC (lambda*)",         "#d29922", 0.50),
    "pred_et": ("prediction · ET (lambda*)",         "#39c5cf", 0.78),
}
MESH_DEC = _b64.b64decode


def mesh3d_figure(mesh_payload, height=560, show=("brain", "gt", "pred", "post")):
    """A plotly Mesh3d figure built from the SAME packed meshes the page ships.

    One trace per layer, so the viewer's own legend turns layers on and off, hovering names the
    structure, and the camera is plotly's problem rather than ours.  `aspectmode='data'` keeps the
    physical proportions (a brain must not be stretched), and the scene has no axes: this is an
    anatomy view, not a chart.
    """
    import plotly.graph_objects as go
    n = [int(v) for v in mesh_payload["n"]]
    layers = mesh_payload["layers"]
    scale = 60000.0 / max(1.0, float(max(n) - 1))
    traces = []
    for key, (label, color, opacity) in MESH_LOOK.items():
        L = layers.get(key)
        if not L:
            continue
        if not any(key.startswith(g) for g in show):
            continue
        v = np.frombuffer(MESH_DEC(L["v"]), np.uint16).reshape(-1, 3).astype(np.float64) / scale
        f = np.frombuffer(MESH_DEC(L["f"]),
                          np.uint32 if L.get("f32") else np.uint16).reshape(-1, 3)
        traces.append(go.Mesh3d(
            x=v[:, 0], y=v[:, 1], z=v[:, 2],
            i=f[:, 0], j=f[:, 1], k=f[:, 2],
            name=label, showscale=False, showlegend=(key != "brain"),
            color=color, opacity=opacity, flatshading=False,
            lighting=dict(ambient=0.55, diffuse=0.85, specular=0.10, roughness=0.9, fresnel=0.05),
            lightposition=dict(x=120, y=180, z=200),
            hovertemplate=label + "<extra></extra>"))
    fig = go.Figure(data=traces)
    fig.update_layout(
        height=height, margin=dict(l=0, r=0, t=26, b=0),
        paper_bgcolor="rgba(0,0,0,0)", plot_bgcolor="rgba(0,0,0,0)",
        scene=dict(xaxis=dict(visible=False), yaxis=dict(visible=False), zaxis=dict(visible=False),
                   aspectmode="data", bgcolor="rgba(0,0,0,0)",
                   camera=dict(eye=dict(x=1.45, y=-1.55, z=0.85), up=dict(x=0, y=0, z=1))),
        legend=dict(orientation="h", yanchor="bottom", y=1.0, xanchor="left", x=0.0,
                    font=dict(color="#c9d1d9", size=10), bgcolor="rgba(0,0,0,0)"),
        hovermode=False, uirevision="keep-the-camera")
    return fig


def plotly3d_html(mesh_payload, height=560):
    """(html, figure) for one case: the interactive viewer, with plotly.js INLINED.

    Kaggle's limit is ~1 MiB for the notebook FILE but ~100 MiB for the file plus its outputs, so a
    library may not sit in a source cell -- but it may sit in an output.  Inlining here is what makes
    the exported page a single self-contained file (no CDN, works offline, works from a USB stick).
    If plotly is missing, the page falls back to its own canvas renderer and says so.
    """
    try:
        fig = mesh3d_figure(mesh_payload, height=height)
        _bundle = PLOTLY_BUNDLE if PLOTLY_BUNDLE in ("inline", "cdn", "directory") else "inline"
        html = fig.to_html(include_plotlyjs=_bundle, full_html=False, div_id="p3d_plot",
                           config={"displayModeBar": False, "responsive": True, "scrollZoom": True})
        return html, fig
    except Exception as e:
        print("      3D viewer: plotly unavailable here (%s: %s) -- the page will use its own "
              "canvas renderer" % (type(e).__name__, e))
        return "", None


def score_case(pid):
    """Rank cases for the demo: things post-processing fixed, missed satellites, low contrast.
    A presentation wants the failure modes on stage, not the lucky cases."""
    z, seg, _ = load_case(pid)
    raw, post = pipeline(z, lam_for(pid))
    gtw = np.isin(seg, safepp.REGIONS["WT"])
    sr = safepp.component_stats(raw, seg, "WT")
    sp = safepp.component_stats(post, seg, "WT")
    se = safepp.component_stats(post, seg, "ET")
    removed = max(0, sr["false_components"] - sp["false_components"])
    c = cnr_of(pid)
    low = int((CUT is not None) and np.isfinite(c) and c <= CUT)
    return 2.0 * removed + 3.0 * se["missed_lesions"] + 1.0 * low, \
        dict(removed=removed, missed=se["missed_lesions"], cnr=c)


def volume_curve(pid):
    """Post-processed volume (mm^3) as a function of the threshold -- the headline number the
    chart interpolates must be the volume of what is actually shown."""
    z, seg, _ = load_case(pid)
    v = []
    for lam in LAMBDAS:
        _, post = pipeline(z, lam)
        v.append(float(np.isin(post, safepp.REGIONS[REG]).sum()))
    return v


def ai_findings(post, seg, raw=None):
    """The three numbers a radiologist actually needs from an auto-segmentation, full resolution.

    `covered`  = share of the radiologist's region that the AI outline already contains (recall).
    `extra`    = share of the AI outline that is not in the radiologist's region (1 - precision).
    `agree`    = Dice.

    These are the same quantities the certificate bounds (missed = 1 - covered, wrong = 1 - extra),
    said in the language of a reading session: "you would keep X % of this outline and delete the
    rest".  `raw` (before post-processing) is optional and only used for the footnote.
    """
    out = {}
    for r in ("WT", "TC", "ET"):
        g = np.isin(seg, safepp.REGIONS[r])
        p = np.isin(post, safepp.REGIONS[r])
        ng, np_ = int(g.sum()), int(p.sum())
        inter = int(np.logical_and(g, p).sum())
        covered = (inter / ng) if ng else float("nan")
        extra = (1.0 - inter / np_) if np_ else float("nan")
        out[r] = dict(gt_mm3=ng, ai_mm3=np_, inter_mm3=inter,
                      covered=covered, extra=extra, agree=float(safepp.dice(p, g)),
                      raw_ai_mm3=(int(np.isin(raw, safepp.REGIONS[r]).sum()) if raw is not None
                                  else None))
    return out


def mask_rle(mask):
    """Run-length encode a boolean mask as (start, length) uint32 pairs over the FLATTENED volume.

    NIfTI stores x fastest, so the flattening is Fortran order -- the same convention the export
    header below declares.  A tumour mask is a few thousand runs, which is why shipping the
    full-resolution delivered mask inside a single HTML page costs ~20 kB instead of 14 MB: the page
    rebuilds the exact full-resolution mask from this when someone asks for the file.
    """
    flat = np.asarray(mask, bool).ravel(order="F")
    nz = np.flatnonzero(flat)
    if nz.size == 0:
        return b"", 0
    breaks = np.flatnonzero(np.diff(nz) > 1)
    starts = np.concatenate(([nz[0]], nz[breaks + 1]))
    ends = np.concatenate((nz[breaks], nz[-1:]))
    pairs = np.stack([starts, ends - starts + 1], axis=1).astype(np.uint32)
    return pairs.tobytes(), int(pairs.shape[0])


def export_payload(mask, seg):
    """Everything the browser needs to write a NIfTI-1 file itself: shape, voxel size, and the
    run-length encoded delivered mask.  The page can then hand the radiologist a file their
    planning system opens -- the point of an auto-segmentation is not a picture, it is a contour."""
    blobby, n_runs = mask_rle(mask)
    return dict(shape=[int(v) for v in seg.shape[::-1]],     # NIfTI order: x, y, z
                voxel_mm=[1.0, 1.0, 1.0],                    # BraTS is 1 mm isotropic
                dtype="uint8", n_runs=n_runs,
                rle=_b64.b64encode(blobby).decode("ascii"))


def region_rows(raw, post, seg, lam=None):
    """WT / TC / ET: truth, model at the certified lambda*, difference and Dice.

    Every number is counted here, full resolution, from the same masks the page draws -- the
    page never recomputes or guesses one.  `raw` is kept so the effect of post-processing on
    each region is visible.
    """
    d_post = safepp.dice_regions(post, seg)
    d_raw = safepp.dice_regions(raw, seg)
    rows = []
    for r in ("WT", "TC", "ET"):
        g = int(np.isin(seg, safepp.REGIONS[r]).sum())
        p = int(np.isin(post, safepp.REGIONS[r]).sum())
        pr = int(np.isin(raw, safepp.REGIONS[r]).sum())
        diff = p - g
        pct = (100.0 * diff / g) if g else float("nan")
        empt = ""
        if g == 0 and p == 0:
            empt = " (both empty &rarr; Dice 1.0 by convention)"
        elif g == 0:
            empt = " (reference empty: Dice 0 by convention)"
        elif p == 0:
            empt = " <b>model predicted nothing here</b> &mdash; the empty-mask convention makes this Dice 0"
        rows.append(
            f"<tr><td><b>{r}</b></td>"
            f"<td>{g:,}</td>"
            f"<td>{p:,}<span class='hint'> ({pr:,} before post-proc.)</span></td>"
            f"<td>{diff:+,}{'' if not np.isfinite(pct) else f' ({pct:+.1f}%)'}</td>"
            f"<td>{d_post[r]:.3f}<span class='hint'> (raw {d_raw[r]:.3f})</span>{empt}</td></tr>")
    return "".join(rows)


# [final 2026-10-10] size-aware lesion counting (matches n2b_lesion_detection.py).
# N2's missed_lesions / false_components count components of ANY size.  On the test split 74 % of the reference
# components are under 100 voxels (median 4, 0.1 % of volume), so those counts describe annotation specks, not
# lesions.  A "meaningful" lesion is >= 1 cm3 (1000 voxels at 1 mm isotropic): a judgement call, change it here.
MEANINGFUL_LESION_VOX = 1000


def lesion_check(post, seg, region=None):
    reg = region or REG
    p = np.isin(post, safepp.REGIONS[reg]); g = np.isin(seg, safepp.REGIONS[reg])
    lg, ng = safepp.components(g); lp, npd = safepp.components(p)
    sg = np.bincount(lg.ravel(), minlength=ng + 1)[1:]
    sp = np.bincount(lp.ravel(), minlength=npd + 1)[1:]
    ov_g = np.bincount(lg[p], minlength=ng + 1)[1:] if ng else np.zeros(0, int)     # predicted voxels inside each reference component
    ov_p = np.bincount(lp[g], minlength=npd + 1)[1:] if npd else np.zeros(0, int)   # reference voxels inside each predicted component
    big = sg >= MEANINGFUL_LESION_VOX
    return dict(n_gt_all=int(ng), missed_all=int((ov_g == 0).sum()),
                n_gt_meaningful=int(big.sum()), missed_meaningful=int(((ov_g == 0) & big).sum()),
                false_all=int((ov_p == 0).sum()),
                false_meaningful=int(((ov_p == 0) & (sp >= MEANINGFUL_LESION_VOX)).sum()))


def _low_evidence(sg):
    """Held-out evidence for the pooled interval on low-contrast scans (empty unless this scan is low-contrast)."""
    try:
        if sg != "low" or TEST_DF is None or CUT is None or "vol_covered" not in TEST_DF or "cnr" not in TEST_DF:
            return ""
        lo = TEST_DF[TEST_DF["cnr"] <= float(CUT)]; hi = TEST_DF[TEST_DF["cnr"] > float(CUT)]
        if not len(lo):
            return ""
        return (f" For orientation: on the held-out scans the pooled interval contained the reference volume in "
                f"{int(lo['vol_covered'].sum())} of {len(lo)} low-contrast scans (and {int(hi['vol_covered'].sum())} of "
                f"{len(hi)} high-contrast). Here it is an average-case range, not a certified one for this scan.")
    except Exception:
        return ""


def engine_rows(raw, post, seg):
    sp = safepp.component_stats(post, seg, REG)
    sr = safepp.component_stats(raw, seg, REG)
    d_r, d_p = safepp.dice_regions(raw, seg), safepp.dice_regions(post, seg)
    lw = safepp.lesionwise_dice(post, seg, REG)[0]
    rows = [
        ("Dice WT / TC / ET", f"{d_r['WT']:.3f} / {d_r['TC']:.3f} / {d_r['ET']:.3f}",
         f"<b>{d_p['WT']:.3f}</b> / <b>{d_p['TC']:.3f}</b> / <b>{d_p['ET']:.3f}</b>"),
        ("lesion-wise Dice (" + REG + ")", "&mdash;", f"{lw:.3f}"),
        ("predicted components", sr["n_pred"], f"{sp['n_pred']}"),
        ("false fragments (any size)", sr["false_components"], f"{sp['false_components']}"),
        ("false lumps >= 1 cm³", lesion_check(raw, seg)["false_meaningful"], f"{lesion_check(post, seg)['false_meaningful']}"),
        ("reference fragments missed (any size)", sr["missed_lesions"], f"{sp['missed_lesions']}"),
        ("lesions >= 1 cm³ missed", lesion_check(raw, seg)["missed_meaningful"], f"{lesion_check(post, seg)['missed_meaningful']}"),
    ]
    return "".join(f"<tr><td>{a}</td><td>{b}</td><td>{c}</td></tr>" for a, b, c in rows)


def _num(x):
    """JSON-safe number: NaN/Inf become None (null in the browser) -- json.dumps would emit a
    bare NaN, which is not valid JSON and renders as 'nan' in a table."""
    try:
        x = float(x)
    except (TypeError, ValueError):
        return None
    return x if np.isfinite(x) else None


def coverage_rows(df):
    """The alpha/beta table.  Rows the calibration could NOT certify are printed as such --
    that is information, not a blank cell."""
    out = []
    for _, r in df.iterrows():
        a = _num(r.alpha); b = _num(r.beta); lam = _num(r.lam)
        key = f"{a if a is not None else 0:.2f}_{b if b is not None else 0:.2f}"
        if lam is None:
            out.append(f'<tr data-k="{key}"><td>{a:.2f}</td><td>{b:.2f}</td>'
                       f'<td colspan=4 style="color:#d29922">not certifiable at this '
                       f'calibration size</td></tr>')
            continue
        out.append(f'<tr data-k="{key}"><td>{a:.2f}</td><td>{b:.2f}</td>'
                   f"<td>{lam:.2f}</td><td>{_num(r.realised_missed):.3f}</td>"
                   f"<td>{_num(r.realised_wrong):.3f}</td>"
                   f"<td>{100 * (_num(r.containment) or 0):.0f}%</td></tr>")
    return "".join(out)


def coverage_data(df):
    return [dict(alpha=_num(r.alpha), beta=_num(r.beta), lam=_num(r.lam),
                 realised_missed=_num(r.realised_missed), realised_wrong=_num(r.realised_wrong),
                 containment=_num(r.containment)) for _, r in df.iterrows()]


def cohort_png(df):
    """Two panels a clinician reads in five seconds: how much tumour was missed per case,
    and how far the reported volume is from the truth, against the certified targets."""
    assert len(df) >= 2, "the cohort figure needs at least two held-out cases"
    import matplotlib
    matplotlib.use("Agg")
    import matplotlib.pyplot as plt
    fig, ax = plt.subplots(1, 2, figsize=(10.4, 3.4), dpi=100)
    ax[0].hist(df.missed_frac, bins=16, color="#f0883e", alpha=.85)
    ax[0].axvline(A, color="#3fb950", lw=1.6, label=f"certified alpha = {A:.2f}")
    ax[0].axvline(df.missed_frac.mean(), color="#e6edf3", lw=1.2, ls="--",
                  label=f"realised mean {df.missed_frac.mean():.3f}")
    ax[0].set_xlabel("missed share of true tumour, per test case")
    ax[0].set_ylabel("cases")
    ax[0].legend(fontsize=7.5)
    err = df.volume_err if "volume_err" in df else df.volume_err_post
    ax[1].scatter(df.volume_gt, err, s=12, color="#58a6ff", alpha=.8)
    ax[1].axhline(0, color="#8b949e", lw=1)
    ax[1].axvline(df.volume_gt.median(), color="#30363d", lw=1, ls=":")
    ax[1].set_xlabel("true volume (mm³)")
    ax[1].set_ylabel("reported − true (mm³)")
    _cov = float(df.vol_covered.mean()) * 100 if "vol_covered" in df else float("nan")
    ax[1].set_title(f"median Dice {df.dice_wt.median():.3f} | interval covered truth in "
                    f"{_cov:.0f}% of cases", fontsize=9)
    for a in ax:
        a.tick_params(labelsize=7.5)
        a.grid(alpha=.15)
    fig.tight_layout()
    buf = io.BytesIO()
    fig.savefig(buf, format="png")
    import matplotlib.pyplot as plt2
    plt2.close(fig)
    return "data:image/png;base64," + base64.b64encode(buf.getvalue()).decode()


def contact(ids):
    out = []
    for pid in ids:
        z, seg, mri = load_case(pid)
        _, post = pipeline(z, lam_for(pid))
        gtw, pw = np.isin(seg, safepp.REGIONS["WT"]), np.isin(post, safepp.REGIONS["WT"])
        if not gtw.any():
            continue
        cz = int(np.round(np.argwhere(gtw).mean(axis=0)[2]))
        base = gray_slice(mri[0], cz)
        rgb = np.dstack([base, base, base])
        ov = np.zeros(gtw.shape[:2], np.uint8)
        rgb[outline(gtw[..., cz]).T] = (63, 255, 90)
        rgb[outline(pw[..., cz]).T] = (90, 170, 255)
        c = cnr_of(pid)
        tag = "contrast unknown" if not np.isfinite(c) else (
            f"low contrast (CNR {c:.1f})" if (CUT is not None and c <= CUT)
            else f"CNR {c:.1f}")
        out.append(dict(id=pid, tag=tag, dice=dice_wt(post, seg),
                        miss=int(lesion_check(post, seg)["missed_meaningful"] > 0),
                        img=to_png(rgb, "RGB")))
    return out


def render_case(pid):
    lam = lam_for(pid)                     # the threshold THIS case is certified at
    _t = {"start": time.perf_counter()}
    z, seg, mri = load_case(pid)
    raw, post = pipeline(z, lam)
    _t["segmentation"] = time.perf_counter()
    gt, raw_w, post_w = seg > 0, raw > 0, post > 0
    em = np.zeros(seg.shape, np.uint8)
    em[raw_w & ~gt] = 1
    em[gt & ~raw_w] = 2
    tum = np.argwhere(gt)
    z0, z1 = int(tum[:, 2].min()) - 4, int(tum[:, 2].max()) + 4
    zs = list(range(max(0, z0), min(seg.shape[2] - 1, z1) + 1))
    if len(zs) > N_SLICES:
        m = len(zs) // 2
        zs = zs[m - N_SLICES // 2: m + N_SLICES // 2]
    zmid = int(np.round(tum.mean(axis=0)[2]))
    zmid = min(max(zmid, zs[0]), zs[-1])
    # per-region sharp layers: the page draws the reference and the model in the SAME region
    # colours on all three planes, instead of "green reference / orange prediction" on the axial
    # stack and region colours on the cross planes
    gtr = {r: np.isin(seg, safepp.REGIONS[r]) for r in ("WT", "TC", "ET")}
    prr = {r: np.isin(raw, safepp.REGIONS[r]) for r in ("WT", "TC", "ET")}
    sl = {k: [] for k in ("flair", "t1ce", "gt", "pred", "post", "err",
                          "gtw", "gtc", "gte", "pw", "pc", "pe")}
    for zz in zs:
        sl["flair"].append(to_png(gray_slice(mri[0], zz)))
        sl["t1ce"].append(to_png(gray_slice(mri[2], zz)))
        sl["gt"].append(mask_layer(gt[..., zz], "gt", 0, 255))       # legacy single-colour
        sl["pred"].append(mask_layer(raw_w[..., zz], "pred"))
        sl["post"].append(mask_layer(post_w[..., zz], "post", 70, 130))
        sl["err"].append(err_layer(em[..., zz]))
        for _k, _r in (("gtw", "WT"), ("gtc", "TC"), ("gte", "ET")):
            sl[_k].append(region_layer(gtr[_r][..., zz], _k, fill=80, edge=135))
        for _k, _r in (("pw", "WT"), ("pc", "TC"), ("pe", "ET")):
            sl[_k].append(region_layer(prr[_r][..., zz], _k, fill=110, edge=165))
    cx, cy, _ = (int(v) for v in np.round(tum.mean(axis=0)))
    # Cross-plane BACKGROUND stacks, so coronal and sagittal get their own slice slider too.
    # Only the background is shipped per slice (sharp); the overlays in those planes are drawn
    # in the browser from the coarse volume, in region colours.
    ortho = {}
    for name, axis, c0 in (("coronal", 1, cy), ("sagittal", 0, cx)):
        full = int(seg.shape[axis])
        i0 = max(0, min(full - 1, c0 - ORTHO_SLICES // 2))
        i1 = min(full, i0 + ORTHO_SLICES)
        idx = list(range(i0, max(i0 + 1, i1)))
        cut = (lambda v, zi, ax=axis: v[:, zi, :] if ax == 1 else v[zi, :, :])
        ortho[name] = dict(
            flair=[to_png(gray_slice(cut(mri[0], zi))) for zi in idx],
            t1ce=[to_png(gray_slice(cut(mri[2], zi))) for zi in idx],
            i0=int(i0), n=len(idx), mid=int(c0))
    gt_v = float(np.isin(seg, safepp.REGIONS[REG]).sum())
    post_v = float(np.isin(post, safepp.REGIONS[REG]).sum())
    lo, hi = safepp.volume_interval(post_v, A, B)
    covered = bool(lo <= gt_v <= hi)
    c = cnr_of(pid)
    ch = cert_for(pid)
    _lam_shown = float(ch["lam"] or LAM)
    stratum = cert_note(pid, ch, c)
    df = COVERAGE[COVERAGE.arm == ARM] if COVERAGE is not None and "arm" in COVERAGE else None
    assert TEST_DF is not None, ("no per-case test table found -- run N2 with RUN_TEST=True and "
                                 "attach its output (test_csvs/)")
    note = (f"Threshold λ* = <b>{_lam_shown:.2f}</b> is the tightest one whose calibration bound stays "
            f"under α = {A:.0%} for missed tumour; it is admissible only because it also stays "
            f"under β = {B:.0%} for wrong prediction.{stratum} Post-processing removes predicted "
            f"components under {(POST.get('min_vox') or {}).get(REG, 0)} voxels, so the volume in the bar is the "
            f"volume of the mask on the left, not of the raw threshold.")
    if not CA.get("valid", True):
        note = (f'<div style="color:#d29922;font-weight:600">ILLUSTRATIVE ONLY &mdash; '
                f'{CA.get("warning")}</div>' + note)
    _cur = ch["curves"] or CA["curves"]
    _t["planes"] = time.perf_counter()
    v3 = vol3d_pack(pid)                   # coarse volumes for the live-λ preview and the 3D view
    _t["vol3d"] = time.perf_counter()
    m3 = mesh3d_pack(pid, v3)              # smoothed isosurfaces for the 3D panel (WebGL)
    _t["mesh3d"] = time.perf_counter()
    _nf = sum(int((L or {}).get("nf", 0)) for L in m3["layers"].values())
    print(f"  3D meshes: {_nf:,} triangle(s) over "
          f"{sum(1 for L in m3['layers'].values() if L)} surface(s)")
    payload = dict(
        defaults=dict(alpha=A, beta=B),
        curves=dict(n_cal=int(ch["n_cal"] or CA["n_cal"]), thresholds=list(map(float, LAMBDAS)),
                    fn_mean=_cur["fn"], fp_mean=_cur["fp"]),
        vol_curve=dict(thr=list(map(float, LAMBDAS)), v=volume_curve(pid)),
        slices=sl, ortho=ortho, vol3d=v3, mesh3d=m3,
        meta=dict(case_id=pid, gt_volume=round(gt_v), post_volume=round(post_v),
                  z0_index=int(zmid), y0=int(np.clip(cy, 0, mri[0].shape[1] - 1)),
                  x0=int(np.clip(cx, 0, mri[0].shape[0] - 1)),
                  vol_wt=f"{int(np.isin(seg, safepp.REGIONS['WT']).sum()):,} mm³ (truth)",
                  vol_tc=f"{int(np.isin(seg, safepp.REGIONS['TC']).sum()):,} mm³ (truth)",
                  vol_et=f"{int(np.isin(seg, safepp.REGIONS['ET']).sum()):,} mm³ (truth)",
                  # the same three regions for the MODEL, so the 3D legend can label which
                  # number belongs to which mask instead of showing truth next to a prediction
                  vol_wt_pred=f"{int(np.isin(post, safepp.REGIONS['WT']).sum()):,} mm³",
                  vol_tc_pred=f"{int(np.isin(post, safepp.REGIONS['TC']).sum()):,} mm³",
                  vol_et_pred=f"{int(np.isin(post, safepp.REGIONS['ET']).sum()):,} mm³",
                  region_rows=region_rows(raw, post, seg, lam=_lam_shown),
                  # clinical panel: what a reading session would take from this outline
                  ai=ai_findings(post, seg, raw),
                  export=export_payload(post, seg),
                  z0=int(zmid - zs[0]), z_i0=int(zs[0]), n_axial=len(zs),
                  z_full=int(seg.shape[2]), lam=float(ch["lam"] or LAM),
                  thr_tc=(POST.get("thr_tc") if POST else None),
                  thr_et=(POST.get("thr_et") if POST else None),
                  cert_scope=ch["scope"], n_cal=int(ch["n_cal"] or CA["n_cal"]),
                  filter_delta=int(SUMMARY.get(ARM, {}).get("filter_delta", 0)),
                  filter_feasible=bool(SUMMARY.get(ARM, {}).get("filter_feasible", False)),
                  note=note, engine_rows=engine_rows(raw, post, seg),
                  coverage_rows=coverage_rows(df) if df is not None else ""),
        coverage=coverage_data(df) if df is not None else [],
        cohort_png=cohort_png(TEST_DF),
        contact=contact([p for p in (SPLIT_JSON.get("test") or [])][:40]),
    )
    _t["payload"] = time.perf_counter()
    return payload, dict(covered=covered, lo=lo, hi=hi, gt=gt_v, post=post_v,
                         t_seconds={k: v - _t["start"] for k, v in _t.items() if k != "start"},
                         t_mark=_t)

# %% [CELL 91] FIX v2.1 -- page overrides + mask-based head shell + AI review + uint16 mesh
# PLACE THIS CELL BETWEEN CELL 7 ("THE PAYLOAD THE GUI READS") AND CELL 8 ("CHOOSE
# CASES, WRITE THE REPORTS"): after the REPORT_TEMPLATE cell (CELL 90 -- in this
# notebook's order the template sits early, right after the shared library) and after
# CELL 7, which defines render_case/mesh3d_pack v1; BEFORE the CELL 8 loop, which
# consumes both to write report_<pid>.html/.json.  "After CELL 90" alone is NOT enough
# in a notebook whose template cell sits above CELL 7, and "at the end" is silently
# useless: CELL 8 would already have written every report with the unfixed code.  The
# placement guard below turns a wrong position into a loud error instead of a silent
# no-op.  Once it sits in the right slot: Run All (or re-run from CELL 5 downwards).
# It
#   * injects a second <script> into the report template (page_fix.js): window/level no
#     longer washes the colour masks, the unsure-voxel dither works on every plane
#     including the sharp axial stack at lambda*, a cursor probe reports reference vs AI
#     vs delivered vs unsure-band under the pointer, the AI review line is shown, and the
#     3D caption warns if the head shell ever fails to enclose the tumour surfaces;
#   * overrides mesh3d_pack so the HEAD surface is an iso-surface of the BRAIN MASK
#     (always encloses the tumour, at any contrast) instead of an intensity level that
#     could collapse into a blob beside the tumour -- and prints a containment check;
#   * adds ai_review() and wraps render_case so meta.ai_review reaches page and sidecar.
# Nothing here changes a certified number.

# ---- placement guard: wrong cell order must fail loudly, never silently ---------
if "REPORT_TEMPLATE" not in globals():
    raise RuntimeError("FIX cell placed too EARLY: REPORT_TEMPLATE (CELL 90) has not run "
                       "yet.  Place this cell after CELL 90 and after CELL 7, before CELL 8.")
if "render_case" not in globals():
    raise RuntimeError("FIX cell placed too EARLY: render_case (CELL 7) has not run yet.  "
                       "Place this cell BETWEEN CELL 7 and CELL 8, then Run All.")
if "TIMINGS" in globals():
    print("!! FIX cell placed too LATE: CELL 8 (the report-writing loop) already ran -- "
          "TIMINGS exists, so the reports on disk were built WITHOUT this fix.  Move this "
          "cell between CELL 7 and CELL 8, then Run All (or re-run from CELL 5 downwards).")

PAGE_FIX_JS = r'''
/* N3 page fix v2 -- injected by the N3 fix cell as a SECOND <script> at the end of the
   report page.  It overrides only what is broken, by name, at call time:

     1. window/level no longer recolours the mask overlays (it used to run every colour
        layer through the same contrast/brightness filter as the scan, so dragging W/L
        washed the reference/prediction masks out -- "the live view is not right");
     2. the unsure-voxel dither is drawn on EVERY plane, including the sharp axial stack
        at the certified lambda* (it used to vanish there, so the tick looked dead);
     3. a cursor probe: move the pointer over any plane and a readout says, for that
        voxel, what the REFERENCE has, what the AI has at the current lambda, whether the
        DELIVERED mask contains it, whether the two agree (agree / extra / missed /
        background) and whether the voxel sits in the unsure band (review purview);
     4. the AI review line (rule-based impression) is shown in the clinical panel and
        appended to the copied report text;
     5. a head-shell containment check: if the 3D head surface does not enclose the
        tumour surfaces, the 3D caption says so instead of silently showing two blobs.

   Every lookup is guarded; what got installed lands in window.__pageFix (and the pure
   helpers in window.__fix) so a headless check or the console can see it.  Nothing here
   changes any number: the probe reads the same coarse volumes the overlays use and says
   so. */
(function (global) {
  "use strict";
  var doc = (typeof document !== "undefined") ? document : null;
  var byId = function (id) { return doc ? doc.getElementById(id) : null; };
  /* script 1 declares S / DATA / V3 / MESHB with const at top level: those are global
     LEXICAL bindings, not window properties, so they must be read by bare name (guarded),
     while its function declarations (drawPlane, drawCover, ...) ARE window properties and
     can be overridden by assignment. */
  function gS() { return (typeof S !== "undefined") ? S : null; }
  function gDATA() { return (typeof DATA !== "undefined") ? DATA : null; }
  function gV3() { return (typeof V3 !== "undefined") ? V3 : null; }
  function gMESHB() { return (typeof MESHB !== "undefined") ? MESHB : null; }
  var installed = [], skipped = [];
  var FIX_VERSION = "v2.1";   /* v2 = page overrides + mask head shell + AI review;
                                 v2.1 = + uint16 mesh re-decode (meshFix) */

  /* the packed-axis convention of the coarse volume, as a local constant so the probe and
     the unsure overlay do not depend on any internal name: rows/cols/fix refer to the
     coarse volume axes (a0 = left-right, a1 = anterior-posterior, a2 = superior-inferior) */
  var PLANE_FIX = { axial: { rows: 1, cols: 0, fix: 2 },
                    coronal: { rows: 2, cols: 0, fix: 1 },
                    sagittal: { rows: 2, cols: 1, fix: 0 } };

  function planeEl(plane) {
    return byId(plane === "axial" ? "view" : (plane === "coronal" ? "cor" : "sag"));
  }
  function sliceState(plane) {
    var S = gS(), DATA = gDATA(), V3 = gV3();
    if (plane === "axial") {
      return { i: S.z | 0, i0: (DATA.meta.z_i0 != null ? DATA.meta.z_i0 : 0),
               n: (DATA.slices.flair || []).length,
               full: (V3 && V3.full) ? V3.full[2] : null };
    }
    var st = (DATA.ortho || {})[plane] || {};
    return { i: (plane === "coronal" ? S.cy : S.cx) | 0, i0: st.i0 || 0,
             n: (st.flair || []).length,
             full: (V3 && V3.full) ? V3.full[plane === "coronal" ? 1 : 0] : null };
  }
  /* the lambda the sliders currently select -- recomputed locally so the probe never
     depends on an internal function name */
  function curLambda() {
    var S = gS(), DATA = gDATA();
    try {
      var n = DATA.curves.n_cal, t = DATA.curves.thresholds, K = t.length;
      var fb = function (i) { return (n / (n + 1)) * DATA.curves.fn_mean[i] + 1 / (n + 1); };
      var gb = function (i) { return (n / (n + 1)) * DATA.curves.fp_mean[i] + 1 / (n + 1); };
      var jFn = null, jFp = null, i;
      for (i = 0; i < K; i++) { if (fb(i) <= S.alpha) jFn = i; }
      for (i = 0; i < K; i++) { if (gb(i) <= S.beta) { jFp = i; break; } }
      return t[(jFn !== null) ? jFn : 0];
    } catch (e) { return gDATA().meta.lam; }
  }

  /* ---------------------------------------------------------------- probe (pure) */
  function probeAt(plane, cssX, cssY, el) {
    var V3 = gV3(), DATA = gDATA();
    if (!V3 || !V3.ready || !el) return null;
    var ax = PLANE_FIX[plane] || PLANE_FIX.axial;
    var s = sliceState(plane);
    var i = Math.max(0, Math.min(s.n - 1, s.i));
    var rows = V3.n[ax.rows], cols = V3.n[ax.cols];
    var r = Math.floor(cssY / Math.max(1, el.clientHeight) * rows);
    var c = Math.floor(cssX / Math.max(1, el.clientWidth) * cols);
    if (r < 0 || c < 0 || r >= rows || c >= cols) return null;
    var fi = (s.full == null) ? 0 :
      Math.max(0, Math.min(V3.n[ax.fix] - 1,
        Math.round((s.i0 + i) * V3.n[ax.fix] / Math.max(1, s.full))));
    var cc = [0, 0, 0]; cc[ax.fix] = fi; cc[ax.rows] = r; cc[ax.cols] = c;
    var q = V3.idx(cc[0], cc[1], cc[2]);
    var lam = curLambda();
    var tTC = (DATA.meta.thr_tc != null) ? DATA.meta.thr_tc : lam;
    var tET = (DATA.meta.thr_et != null) ? DATA.meta.thr_et : lam;
    var gw = !!(V3.gtw && V3.gtw[q] > 127), gc = !!(V3.gtc && V3.gtc[q] > 127), ge = !!(V3.gte && V3.gte[q] > 127);
    var pw = !!(V3.wt && V3.wt[q] / 255 >= lam);
    var pc = pw && !!(V3.tc && V3.tc[q] / 255 >= tTC);
    var pe = pc && !!(V3.et && V3.et[q] / 255 >= tET);
    var del = !!((V3.postw && V3.postw[q] > 127) || (V3.postc && V3.postc[q] > 127) || (V3.poste && V3.poste[q] > 127));
    var p = V3.wt ? V3.wt[q] / 255 : 0;
    return { plane: plane, slice: i + 1, n: s.n, r: r, c: c,
             gw: gw, gc: gc, ge: ge, pw: pw, pc: pc, pe: pe, del: del, p: p, lam: lam,
             unc: Math.abs(p - lam) <= Math.max(0.10, 0.35 * lam) };
  }
  function probeText(q) {
    if (!q) return "";
    var ref = q.ge ? "WT+TC+ET (enhancing core)" : q.gc ? "WT+TC (tumour core)" :
              q.gw ? "WT (oedema/edge)" : "no reference here";
    var ai = q.pe ? "WT+TC+ET" : q.pc ? "WT+TC" : q.pw ? "WT" : "outside";
    var rel = (q.pw && q.gw) ? "agree: inside both outlines" :
              (q.pw && !q.gw) ? "EXTRA: AI only, review first" :
              (!q.pw && q.gw) ? "MISSED: reference only, AI off" :
              "background: inside neither";
    return "cursor · " + q.plane + " slice " + q.slice + "/" + q.n + " · row " + q.r + " col " + q.c +
      " · reference: " + ref +
      " · AI at λ=" + q.lam.toFixed(2) + ": " + ai + " (p=" + q.p.toFixed(2) + ")" +
      " · delivered mask: " + (q.del ? "inside" : "outside") +
      " · " + rel + (q.unc ? " · IN UNSURE BAND (review purview)" : "");
  }

  /* ---------------------------------------------------- head-shell containment */
  function headCheck() {
    var M = gMESHB();
    if (!M || !M.layers) return null;
    var bb = {}, k, i, a;
    for (k in M.layers) {
      var L = M.layers[k]; if (!L || !L.pos) continue;
      var lo = [1e9, 1e9, 1e9], hi = [-1e9, -1e9, -1e9];
      for (i = 0; i < L.nv; i++) for (a = 0; a < 3; a++) {
        var v = L.pos[i * 3 + a]; if (v < lo[a]) lo[a] = v; if (v > hi[a]) hi[a] = v;
      }
      bb[k] = [lo, hi];
    }
    var head = bb.brain; if (!head) return { head: false, bad: [] };
    var bad = [];
    for (k in bb) {
      if (k === "brain") continue;
      var b = bb[k];
      for (a = 0; a < 3; a++) {
        if (b[0][a] < head[0][a] - 1.5 || b[1][a] > head[1][a] + 1.5) { bad.push(k); break; }
      }
    }
    return { head: true, bad: bad };
  }

  /* ------------------------------------------------- FIX v2.1: mesh vertices are UINT16
     N3's mesh3d packs vertices as raw 16-bit LE triples with pos_cells = v / scale.  Read as
     Int16, every coordinate above 32767/scale (~51.88 cells for the real payloads) WRAPS to
     the far side of the volume -- measured on report_BraTS20_Training_095.json: unsigned
     decode puts 100% of gt_wt vertices within 2 cells of the reference surface, signed only
     40%.  That wrap is the "tumour blob sitting beside the head blob" artefact.  This
     re-decode is idempotent: if the page already decoded unsigned it rewrites identical
     values.  Faces need no patch here: every layer's nv stays below 32768, so signed and
     unsigned index reads coincide. */
  function meshFix() {
    var DATA = gDATA(), M = gMESHB();
    if (!M || !M.layers) return -1;
    var src = (DATA && DATA.mesh3d && DATA.mesh3d.layers) || null;
    if (!src) return -1;
    var fixed = 0, k;
    for (k in src) {
      var S = src[k], D = M.layers[k];
      if (!S || typeof S.v !== "string" || !S.scale || !D || !D.pos || !D.nv) continue;
      if (D.pos.length < D.nv * 3) continue;
      try {
        var bin = atob(S.v), bytes = new Uint8Array(bin.length), i2;
        for (i2 = 0; i2 < bin.length; i2++) bytes[i2] = bin.charCodeAt(i2);
        var u16 = new Uint16Array(bytes.buffer, bytes.byteOffset, (bytes.length / 2) | 0);
        if (u16.length < D.nv * 3) continue;
        for (i2 = 0; i2 < D.nv * 3; i2++) D.pos[i2] = u16[i2] / S.scale;
        fixed++;
      } catch (e) { }
    }
    return fixed;
  }

  if (!doc) {   /* headless: export the pure helpers and stop */
    global.__fix = { probeAt: probeAt, probeText: probeText, curLambda: curLambda,
                     headCheck: headCheck, meshFix: meshFix, PLANE_FIX: PLANE_FIX,
                     installed: installed, skipped: skipped, version: FIX_VERSION };
    return;
  }

  /* ============================ 1. window/level must not touch the masks ====== */
  if (typeof global.drawCover === "function" && typeof global.img === "function" &&
      typeof global.drawSharp === "function") {
    var _dc = global.drawCover, _nofilter = 0;
    global.drawCover = function (ctx, uri, W, H, keep) {
      if (keep === true || _nofilter) {
        var im = global.img(uri);
        if (im && (im.__ready || im.naturalWidth)) ctx.drawImage(im, 0, 0, W, H);
        return;
      }
      return _dc(ctx, uri, W, H);
    };
    var _ds = global.drawSharp;
    global.drawSharp = function (ctx, i, W, H) {
      _nofilter++; try { return _ds(ctx, i, W, H); } finally { _nofilter--; }
    };
    installed.push("wl-does-not-touch-masks");
  } else skipped.push("wl-does-not-touch-masks");

  /* ============================ 2. unsure dither on every plane =============== */
  function uncOverlay(plane) {
    if (typeof global.coarseMask !== "function" || !gV3() || !gV3().ready) return null;
    var s = sliceState(plane);
    var iAbs = s.i0 + Math.max(0, Math.min(s.n - 1, s.i));
    var lam = curLambda();
    return global.coarseMask(plane, iAbs, lam, { gt: 0, raw: 0, post: 0, err: 0, unc: 1 });
  }
  var PROBE_LINE = null, LAST = null;
  function refreshProbe(plane) {
    if (!PROBE_LINE || !LAST || (plane && LAST.plane !== plane)) return;
    var q = probeAt(LAST.plane, LAST.x, LAST.y, LAST.el);
    if (q) PROBE_LINE.textContent = probeText(q);
  }
  if (typeof global.drawPlane === "function") {
    var _dp = global.drawPlane;
    global.drawPlane = function (plane) {
      _dp(plane);
      if (gS() && gS().unc) {           /* the dither, also on the sharp axial stack */
        var u = uncOverlay(plane), el = planeEl(plane);
        if (u && el) {
          var ctx = el.getContext("2d");
          ctx.imageSmoothingEnabled = false;
          ctx.drawImage(u, 0, 0, el.width, el.height);
          ctx.imageSmoothingEnabled = true;
        }
      }
      refreshProbe(plane);
    };
    installed.push("unc-on-all-planes");
  } else skipped.push("unc-on-all-planes");

  /* ============================ 3. cursor probe wiring ======================== */
  (function () {
    var line = doc.createElement("div");
    line.id = "probeLine"; line.className = "hint";
    line.style.cssText = "min-height:18px;color:#9fb0c0;font-size:11.5px;margin-top:6px";
    line.textContent = "cursor probe: move the pointer over any plane — reference vs AI vs delivered vs unsure band";
    var anchor = byId("view");
    var panel = (anchor && anchor.closest) ? anchor.closest(".panel") : null;
    (panel || byId("cockpit") || doc.body).appendChild(line);
    PROBE_LINE = line;
    var planes = [["view", "axial"], ["cor", "coronal"], ["sag", "sagittal"]];
    planes.forEach(function (pr) {
      var el = byId(pr[0]); if (!el) return;
      el.style.cursor = "crosshair";
      el.addEventListener("pointermove", function (ev) {
        var rect = el.getBoundingClientRect();
        LAST = { plane: pr[1], x: ev.clientX - rect.left, y: ev.clientY - rect.top, el: el };
        var q = probeAt(pr[1], LAST.x, LAST.y, el);
        if (q) line.textContent = probeText(q);
      });
      el.addEventListener("pointerleave", function () {
        if (LAST && LAST.plane === pr[1]) LAST = null;
      });
    });
    installed.push("cursor-probe");
  })();

  /* ============================ 4. AI review line ============================= */
  (function () {
    var t = gDATA() && gDATA().meta && gDATA().meta.ai_review;
    if (!t) { skipped.push("ai-review"); return; }
    var panel = byId("aiPanel"); if (!panel) { skipped.push("ai-review"); return; }
    var d = doc.createElement("div");
    d.className = "certmini";
    d.innerHTML = "<b>AI review</b> (rule-based impression from counted volumes, not a diagnosis): " + t;
    panel.appendChild(d);
    installed.push("ai-review");
    if (typeof global.reportText === "function") {
      var _rt = global.reportText;
      global.reportText = function () {
        return _rt() + "\n  AI review (rule-based, not a diagnosis): " + t;
      };
    }
  })();

  /* ============================ 4b. .nii export (header + axis order) ===========
     [merge 2026-10-09]  The report template's niftiBytes() had two defects, both visible in nibabel:
       * header: dim[] was written as int32 at offset 40 (dim is int16), srow offsets wrong, qform/sform
         both claimed valid -> the file is not a readable NIfTI;
       * shape: export.shape is (Z,Y,X) in C order -- which IS x-fastest NIfTI order -- but it was read
         as (nx,ny,nz).  The data was right; the dimensions were reversed.
     This override reads shape as (Z,Y,X), writes NIfTI dims (X,Y,Z), and an identity-orientation sform
     scaled by the voxel size (the cache is the cropped brain box, so no world position is claimed). */
  function niiFromExport(ex) {
    var sh = ex.shape, nx = sh[2], ny = sh[1], nz = sh[0];
    var bin = atob(ex.rle), u8 = new Uint8Array(bin.length), i;
    for (i = 0; i < bin.length; i++) u8[i] = bin.charCodeAt(i);
    var dvr = new DataView(u8.buffer), vol = new Uint8Array(nx * ny * nz), total = vol.length;
    for (i = 0; i + 8 <= u8.length; i += 8) {
      var s0 = dvr.getInt32(i, true), ln = dvr.getInt32(i + 4, true);
      if (s0 < 0 || ln < 0 || s0 + ln > total) throw new Error("export runs fall outside the volume");
      vol.fill(1, s0, s0 + ln);
    }
    var mm = ex.voxel_mm || [1, 1, 1];
    var H = new ArrayBuffer(352), dv = new DataView(H);
    dv.setInt32(0, 348, true);
    dv.setInt16(40, 3, true); dv.setInt16(42, nx, true); dv.setInt16(44, ny, true);
    dv.setInt16(46, nz, true); dv.setInt16(48, 1, true);
    dv.setInt16(70, 2, true); dv.setInt16(72, 8, true);                     /* uint8 */
    dv.setFloat32(76, 1, true);
    dv.setFloat32(80, mm[0], true); dv.setFloat32(84, mm[1], true); dv.setFloat32(88, mm[2], true);
    dv.setFloat32(108, 352, true); dv.setFloat32(112, 1, true);
    dv.setInt16(252, 0, true); dv.setInt16(254, 1, true);                   /* qform off, sform on */
    dv.setFloat32(280, mm[0], true); dv.setFloat32(300, mm[1], true); dv.setFloat32(320, mm[2], true);
    dv.setUint8(344, 0x6E); dv.setUint8(345, 0x2B); dv.setUint8(346, 0x31);
    var out = new Uint8Array(352 + vol.length);
    out.set(new Uint8Array(H), 0); out.set(vol, 352);
    return { bytes: out, nx: nx, ny: ny, nz: nz };
  }
  if (typeof global.niftiBytes === "function") {
    global.niftiBytes = function () { return niiFromExport(gDATA().meta.export); };
    installed.push("nii-export-fixed");
  } else skipped.push("nii-export-fixed");
  global.__niiFromExport = niiFromExport;

  /* ============================ 4c. contact-sheet wording ======================= [final 2026-10-10]
     N3 now flags a scan only when a lesion of >= 1 cm3 is missed: sub-1 cm3 fragments are annotation specks that
     dominate raw component counts.  The template's captions still say "satellite missed" / "all lesions found". */
  (function () {
    var c = byId("contact"); if (!c) return;
    Array.prototype.forEach.call(c.querySelectorAll("figcaption span"), function (s) {
      if (s.textContent === "satellite missed") s.textContent = "lesion \u2265 1 cm\u00b3 missed";
      else if (s.textContent === "all lesions found") s.textContent = "all lesions \u2265 1 cm\u00b3 found";
    });
    installed.push("contact-wording");
  })();

  /* ============================ 5. mesh redecode + head containment + status ==== */
  var nFixed = meshFix();
  if (nFixed > 0) installed.push("mesh-u16-redecode(" + nFixed + " layers)");
  var hc = null;
  try {
    hc = headCheck();
    if (hc && hc.head && hc.bad.length) {
      var n = byId("glNote");
      if (n) n.textContent += "  ·  WARNING: the head shell does not enclose " +
        hc.bad.join(", ") + " — re-run N3 with the fix cell";
    }
  } catch (e) { hc = { error: String(e && e.message) }; }
  if (typeof global.render === "function") {
    var _r = global.render;
    global.render = function () { var o = _r(); refreshProbe(); return o; };
    /* script 1 already painted the page with the wrapped vertices: repaint once, corrected */
    if (nFixed > 0) { try { global.render(); } catch (e) { } }
  }
  /* visible proof of which fix version the report page is running */
  if (typeof document !== "undefined" && document.body && !document.getElementById("fixBadge") &&
      typeof document.body.insertAdjacentHTML === "function") {
    document.body.insertAdjacentHTML("afterbegin",
      "<div id=\"fixBadge\" style=\"position:fixed;top:6px;right:8px;z-index:99;" +
      "font:600 10px/1 ui-monospace,Menlo,Consolas,monospace;color:#68d391;background:#10241a;" +
      "border:1px solid #22543d;border-radius:999px;padding:4px 9px\">page fix " + FIX_VERSION +
      " &middot; uint16 mesh decode &middot; mask-based head shell</div>");
  }
  global.__pageFix = { installed: installed, skipped: skipped, head: hc, meshLayersFixed: nFixed,
                       version: FIX_VERSION };
  global.__fix = { probeAt: probeAt, probeText: probeText, curLambda: curLambda,
                   headCheck: headCheck, meshFix: meshFix, PLANE_FIX: PLANE_FIX,
                   installed: installed, skipped: skipped, version: FIX_VERSION };
})(typeof window !== "undefined" ? window : globalThis);

'''

assert REPORT_TEMPLATE.count("</body>") == 1, "template shape changed -- inject by hand"
REPORT_TEMPLATE = REPORT_TEMPLATE.replace(
    "</body>", "<script>\n" + PAGE_FIX_JS + "\n</script>\n</body>", 1)
print("page fix v2 injected into the template: %d chars of override JS" % len(PAGE_FIX_JS))

# ---------------------------------------------------------------------------
# N3 payload fix v2 -- Python half.  Redefines mesh3d_pack (the head surface),
# adds ai_review(), and wraps render_case so the review reaches the page.
# Lives in the fix cell; overrides the CELL 7 definitions by name.
# ---------------------------------------------------------------------------

def mesh3d_pack(pid, v3=None):
    """Smooth surfaces for the 3D panel -- v2: the HEAD is an iso-surface of the
    BRAIN MASK (nonzero MRI), not of the MRI intensity.

    Why this changed: an intensity iso-surface (level 0.18 of the normalised FLAIR)
    is at the mercy of the contrast of that one scan.  On a scan whose parenchyma
    normalises below the level, the "head" surface collapses to whatever is bright
    -- often the oedema sleeve or a stray patch -- and the 3D view then shows the
    tumour shell and a head-ish blob as TWO SEPARATE BLOBS, in the plotly view and
    in the canvas view alike (both read the same meshes).  The nonzero mask is the
    skull-stripped brain outline itself: it always encloses the tumour, on every
    scan, at every contrast.

    Two more guarantees, checked out loud instead of silently:
      * the head crop box is the UNION of the head box and every tumour layer box,
        so the head surface can never be extracted from a crop that misses a lesion;
      * after pooling, the brain mask must be solid inside every tumour layer box;
        if it is not (a mask/contrast pathology), the run prints HEAD CHECK: False
        rather than handing over a head that does not contain the tumour.
    """
    v3 = v3 or vol3d_pack(pid)
    z, seg, mri = load_case(pid)
    shape = tuple(int(x) for x in seg.shape)
    tgt = _coarse_shape(shape, MESH_N)
    lam = float(lam_for(pid))
    pw = z["p_wt"] >= lam
    ptc = np.logical_and(z["p_tc"] >= lam, pw)
    pet = np.logical_and(z["p_et"] >= lam, ptc)
    _, post = pipeline(z, lam)
    fields = dict(pred_wt=pw, pred_tc=ptc, pred_et=pet,
                  gt_wt=np.isin(seg, safepp.REGIONS["WT"]),
                  gt_tc=np.isin(seg, safepp.REGIONS["TC"]),
                  gt_et=np.isin(seg, safepp.REGIONS["ET"]),
                  post_wt=np.isin(post, safepp.REGIONS["WT"]),
                  post_tc=np.isin(post, safepp.REGIONS["TC"]),
                  post_et=np.isin(post, safepp.REGIONS["ET"]))
    meshes, boxes = {}, {}
    for k, m in fields.items():
        m = np.asarray(m)
        box = _bbox_of(m, shape, tgt, level=0.5)
        boxes[k] = box
        meshes[k] = None if box is None else _mesh_one(m.astype(np.float32), shape, tgt, box,
                                                       MESH_CELLS)
    # -- the head: geometry from the brain MASK, blurred like every other layer so the
    #    surface is smooth; level 0.5 of the blurred mask = the brain outline
    head = (np.abs(mri[0]) > 1e-6).astype(np.float32)
    hbox = _bbox_of(head, shape, tgt, level=0.5, pad_frac=0.03, pad_min=2.0)
    for k, b in boxes.items():                       # union with every tumour layer box
        if b is None:
            continue
        if hbox is None:
            hbox = b
            continue
        hbox = ([min(a, c) for a, c in zip(hbox[0], b[0])],
                [max(a, c) for a, c in zip(hbox[1], b[1])])
    meshes["brain"] = None if hbox is None else \
        _mesh_one(head, shape, tgt, hbox, MESH_BRAIN_CELLS, level=0.5, sigma=1.6)
    # -- containment self-check, printed: the failure this replaces was silent.
    #    The right question is voxel-wise, not box-wise: every coarse voxel where a
    #    tumour layer lives must also be inside the head mask (a padded box corner may
    #    legitimately fall outside the brain -- that is air, not a misregistration).
    hlow = _pool3(head, tgt, "mean") > 0.5
    ok = True
    for k, m in fields.items():
        tlow = _pool3(np.asarray(m, np.float32), tgt, "mean") > 0.5
        if bool(tlow.any()) and not bool(hlow[tlow].all()):
            ok = False
            print("      HEAD CHECK: the head mask does not contain every %s voxel" % k)
    print("      head shell: mask-based (not intensity); encloses every tumour layer: %s"
          % ok)
    dens = []
    for k in ("gt_wt", "gt_tc", "gt_et"):
        bx = boxes.get(k)
        if bx is not None:
            dens.append(int(min(MESH_CELLS, max(_extent(bx, shape, tgt)))))
    if dens:
        print("      mesh density: %d..%d cell(s) across the reference surfaces (cap %d)"
              % (min(dens), max(dens), MESH_CELLS))
    return dict(n=list(map(int, tgt)), cells=int(MESH_CELLS), layers=_mesh_pack(meshes, tgt))


# ---------------------------------------------------------------------------
# [merge 2026-10-09] Automatic description: built from the AI OUTLINE and the scan's contrast only.
#
# The earlier ai_review() counted lesions, enhancement and volume from the REFERENCE segmentation.  That
# cannot exist in a reading session -- there the radiologist has no reference yet -- so a description that
# needs it describes the evaluation set, not the tool.  This one uses only what the tool has: the
# delivered outline, its probabilities and the CNR.  The comparison with the reference is kept, but as a
# separate, labelled evaluation sentence.  Every line ships the rule behind it.
DESC_MIN_LESION_VOX = 1000     # a "lesion" = connected whole-tumour component >= 1 cm3 ...
DESC_REL_LESION = 0.05         # ... and >= 5 % of the largest one
DESC_SIZE_CM3 = (10.0, 60.0)   # descriptive bins: small / medium / large (not clinical thresholds)


def case_descriptor(pid, z, post, lam):
    wt = np.isin(post, safepp.REGIONS["WT"])
    v_wt = int(wt.sum())
    v_tc = int(np.isin(post, safepp.REGIONS["TC"]).sum())
    v_et = int(np.isin(post, safepp.REGIONS["ET"]).sum())
    dis = ("Rule-based description of the AI outline and the scan contrast. It cannot tell histology or "
           "grade and is not a diagnosis; the radiologist's reading decides.")
    if v_wt == 0:
        return dict(headline="the model outlined no tumour on this scan", items=[], disclaimer=dis)
    lab, n = safepp.components(wt)
    sizes = np.bincount(lab.ravel())[1:]
    big = int(sizes.max())
    lesions = sorted([int(s) for s in sizes if s >= max(DESC_MIN_LESION_VOX, DESC_REL_LESION * big)],
                     reverse=True)
    frag = int(n) - len(lesions)
    items = []
    if len(lesions) >= 2:
        foc = "multifocal"
        items.append(dict(k="Focality", tone="warn",
                          v="%d separate lesions (%s cm3)" % (len(lesions), ", ".join("%.1f" % (s / 1000.0) for s in lesions[:4])),
                          why="lesion = connected whole-tumour component >= max(1 cm3, 5 %% of the largest); "
                              "%d smaller fragment(s) not counted. Check each focus." % frag))
    else:
        foc = "single-focus"
        items.append(dict(k="Focality", tone="ok",
                          v="one main lesion" + (" (+%d fragment(s))" % frag if frag else ""),
                          why="lesion = connected whole-tumour component >= max(1 cm3, 5 % of the largest)."))
    cm3 = v_wt / 1000.0
    size = "small" if cm3 < DESC_SIZE_CM3[0] else ("medium" if cm3 < DESC_SIZE_CM3[1] else "large")
    items.append(dict(k="Size", tone="info", v="%s, whole tumour %.1f cm3" % (size, cm3),
                      why="bins: < %d cm3 small, < %d cm3 medium, else large (descriptive only)." % DESC_SIZE_CM3))
    cnr = cnr_of(pid)
    if np.isfinite(cnr) and CUT is not None:
        low = cnr <= float(CUT)
        con = "low-contrast" if low else "high-contrast"
        items.append(dict(k="Contrast", tone="warn" if low else "ok",
                          v="%s scan (CNR %.1f, cohort cut %.1f)" % (con, cnr, float(CUT)),
                          why="CNR = tumour-to-surround contrast over noise; the cut splits the cohort into the two "
                              "strata the certificate is calibrated on. Low contrast = softer boundary."))
    else:
        con = "contrast unknown"
        items.append(dict(k="Contrast", tone="info", v="not available for this case", why=""))
    if v_et > 0:
        items.append(dict(k="Enhancement", tone="info",
                          v="enhancing tumour %.1f cm3 (%.0f %% of the whole tumour)" % (v_et / 1000.0, 100.0 * v_et / v_wt),
                          why="ET label of the delivered mask; non-enhancing core %.1f cm3." % (max(0, v_tc - v_et) / 1000.0)))
    else:
        items.append(dict(k="Enhancement", tone="warn", v="no enhancing tumour in the outline",
                          why="the delivered mask has no ET voxels (absent, or below the frozen ET size floor)."))
    p = np.asarray(z["p_wt"], np.float32)
    band = max(0.10, 0.35 * float(lam))
    cand = int((p >= float(lam) - band).sum())
    unsure = float((np.abs(p - float(lam)) <= band).sum()) / max(1, cand)
    items.append(dict(k="Boundary certainty", tone="ok" if unsure < 0.15 else "warn",
                      v="%.0f %% of the plausible tumour voxels sit near the cut-off" % (100.0 * unsure),
                      why="near = within max(0.10, 35 %% of lambda*) of lambda* = %.2f; the 'unsure voxels' layer shows where."
                          % float(lam)))
    return dict(headline="%s, %s, %s outline" % (foc, size, con), items=items, disclaimer=dis,
                rules=dict(min_lesion_vox=DESC_MIN_LESION_VOX, rel=DESC_REL_LESION, size_cm3=list(DESC_SIZE_CM3)))


def ai_review(post, seg, raw, cnr, desc=None):
    """Text form of the description (copy-report button and the app's fallback) plus ONE labelled evaluation
    sentence against the reference.  [final 2026-10-10] lesions are counted by size: fragments under 1 cm3 are
    reported separately, because they dominate raw component counts and carry almost no volume."""
    parts = []
    if desc:
        parts.append(desc["headline"] + ": " + "; ".join("%s %s" % (i["k"].lower(), i["v"]) for i in desc["items"]))
    g, p = np.isin(seg, safepp.REGIONS["WT"]), np.isin(post, safepp.REGIONS["WT"])
    if int(g.sum()):
        inter = int(np.logical_and(g, p).sum())
        lc = lesion_check(post, seg, "WT")
        small = lc["n_gt_all"] - lc["n_gt_meaningful"]
        small_found = (lc["n_gt_all"] - lc["missed_all"]) - (lc["n_gt_meaningful"] - lc["missed_meaningful"])
        parts.append("Against the reference (evaluation only, not available in a real reading): the outline contains "
                     "%.0f %% of the reference whole tumour and %.0f %% of the outline is not in it; lesions >= 1 cm3: "
                     "%d of %d found (any overlap); plus %d reference fragment(s) under 1 cm3, %d of them touched"
                     % (100.0 * inter / int(g.sum()), 100.0 * (1.0 - inter / max(1, int(p.sum()))),
                        lc["n_gt_meaningful"] - lc["missed_meaningful"], lc["n_gt_meaningful"], small, small_found))
    return ".  ".join(parts) + ".  Rule-based, not a histological or diagnostic claim."


_render_case_v1 = render_case


def render_case(pid):
    """CELL 7's render_case plus meta.describe (structured) and meta.ai_review (its text form)."""
    payload, info = _render_case_v1(pid)
    z, seg, _m = load_case(pid)
    _lam = float(lam_for(pid))
    _raw, _post = pipeline(z, _lam)
    _d = case_descriptor(pid, z, _post, _lam)
    payload["meta"]["describe"] = _d
    payload["meta"]["ai_review"] = ai_review(_post, seg, _raw, cnr_of(pid), desc=_d)
    payload["meta"]["stratum"] = stratum_of(pid)                   # [final 2026-10-10] low / high / unknown: lets the app flag pooled-on-low-contrast
    payload["meta"]["lesion_min_vox"] = int(MEANINGFUL_LESION_VOX)   # [final 2026-10-10] the contact-sheet "miss" flag means: a lesion this big was missed
    return payload, info


print("FIX v2.1 ready: mask-based head shell + containment check | AI review | page overrides | uint16 mesh re-decode (meshFix) + visible version badge")


# %% [CELL 8] CHOOSE CASES, WRITE THE REPORTS ---------------------------------
# a case is usable only if BOTH its probabilities and its cached ground truth are here:
# an attached cache can legitimately be a subset (shards), and the report must skip the
# rest rather than die on the first missing .npy
pool = [p for p in (SPLIT_JSON.get(SPLIT) or [])
        if os.path.exists(os.path.join(ARMDIR, f"{p}.npz"))
        and os.path.exists(bratsx.cache_file(CACHE_BASE, p, "seg"))]
if not pool:
    pool = sorted(os.path.basename(f)[:-4] for f in glob.glob(os.path.join(ARMDIR, "*.npz")))
assert pool, f"no probability files in {ARMDIR}"
if CASE_ID:
    chosen = [CASE_ID]
else:
    ranked = sorted(((score_case(p)[0], p) for p in pool), reverse=True)
    chosen = [p for _, p in ranked[:N_REPORT]]
    print("auto-picked:", [(p, score_case(p)[1]) for p in chosen])

os.makedirs(os.path.join(WORK, "reports"), exist_ok=True)
TIMINGS = []
for pid in chosen:
    _t0 = time.perf_counter()
    payload, info = render_case(pid)
    out = os.path.join(WORK, "reports", f"report_{pid}.html")
    _t_p3d = time.perf_counter()
    _p3d_html, _p3d_fig = plotly3d_html(payload.get("mesh3d") or {}, height=560)
    _n_traces = 0 if _p3d_fig is None else len(_p3d_fig.data)
    print("      3D viewer: %s" % ("plotly, %d interactive mesh trace(s), %d kB inlined"
                                   % (_n_traces, len(_p3d_html) / 1e3) if _p3d_html
                                   else "the page's own canvas renderer (no plotly.js)"))
    # one payload, one contract: the page gets the SAME sanitised JSON as the sidecar, so a value
    # that is null in report_<pid>.json is null in the page too (json.dumps alone would write a bare
    # NaN, which is legal JS but shows up as "NaN" the moment a panel formats it)
    _html = (REPORT_TEMPLATE.replace("@@JSON@@", json.dumps(jsonable(payload)))
             .replace("@@PLOTLY@@", _p3d_html))
    open(out, "w").write(_html)
    # strict, portable JSON: see jsonable() -- a bare NaN would make the sidecar unparseable
    json.dump(jsonable(payload), open(out[:-5] + ".json", "w"))
    _t_write = time.perf_counter() - _t0
    _st = dict(info.get("t_seconds") or {})
    _budget = dict(segmentation=_st.get("segmentation", float("nan")),
                   planes=_st.get("planes", float("nan")) - _st.get("segmentation", 0.0),
                   vol3d=_st.get("vol3d", float("nan")) - _st.get("planes", 0.0),
                   payload=_st.get("payload", float("nan")) - _st.get("vol3d", 0.0),
                   html_write=_t_write)
    TIMINGS.append(_budget)
    print(f"{pid}: report build {_t_write:.1f}s  "
          f"(segmentation {_budget['segmentation']:.1f} | 2D planes {_budget['planes']:.1f} | "
          f"3D bundle {_budget['vol3d']:.1f} | payload {_budget['payload']:.1f} | "
          f"HTML {_budget['html_write']:.1f})")
    m = payload["meta"]
    print(f"\n{pid}: [{m['cert_scope']} certificate, lambda*={m['lam']:.2f}, n_cal={m['n_cal']}]")
    print(f"{pid}: {REG} volume {m['post_volume']} mm³ | truth {info['gt']:.0f} | certified "
          f"[{info['lo']:.0f}, {info['hi']:.0f}] | "
          f"{'COVERED' if info['covered'] else 'NOT covered'} | "
          f"{os.path.getsize(out)/1e3:.0f} kB -> {out}")
    print(f"      image dir {os.path.dirname(out)}: download the .html and open it in a browser")
    # the same viewer, inline in THIS notebook cell -- one interactive 3D model, two places, and it
    # survives in the saved version's output (the ~100 MiB output budget, not the 1 MiB source one)
    if _p3d_html:
        try:
            from IPython.display import display, HTML as _HTML
            display(_HTML('<div style="font:12px system-ui;color:#8b949e">%s — interactive: drag to '
                          'rotate, scroll to zoom, click a legend entry to hide a layer</div>'
                          % pid))
            display(_HTML(_p3d_html))
        except Exception as _e:
            print(f"      (inline notebook view skipped: {_e})")

if TIMINGS:
    import statistics as _stat
    _tot = [sum(t.values()) for t in TIMINGS]
    print(f"\nTIME FOR ONE PATIENT (report stage only, {len(TIMINGS)} case(s)): "
          f"{_stat.mean(_tot):.1f}s each "
          f"[segmentation {_stat.mean([t['segmentation'] for t in TIMINGS]):.1f} | "
          f"planes {_stat.mean([t['planes'] for t in TIMINGS]):.1f} | "
          f"3D bundle {_stat.mean([t['vol3d'] for t in TIMINGS]):.1f} | "
          f"HTML {_stat.mean([t['html_write'] for t in TIMINGS]):.1f}]")
    print("   inference is NOT in this number -- N2 prints it as s/case; the delivered "
          "configuration is the ensemble, so that cost is shared by the arms it averages.")


# %% [CELL 9] NEXT -----------------------------------------------------------
print("""
NEXT
  * each report is one self-contained HTML file: no CDN, no network, no external images.
  * the page is interactive: an axial/coronal/sagittal stack with slice sliders, a live
    alpha/beta preview that re-thresholds the mask in the browser, and a 3D view of the
    prediction coloured by region (drag to rotate, wheel to zoom, double-click to reset).
    N3_PAGE_GUIDE.md in kaggle_v2_scripts.zip explains every control in order.
    Download it from the Kaggle output pane (or attach the 'reports' folder as an output
    dataset and open it from the file browser).
  * every number in the report is read from what N2 froze -- certificates.json,
    frozen_postproc.json, coverage_table.csv -- so the picture cannot drift from the
    statistics.  If you change FORCE_ALPHA / FORCE_BETA in N2, re-run N2 then N3.
  * the curves drawn are per-threshold WORST-CASE ENVELOPES over the calibration cases
    (that is what makes the selected threshold valid across all thresholds at once), so
    they sit above the average risk; the coverage table reports what actually happened.
  * for the presentation, render one case per failure mode:
        CASE_ID='...'  a false fragment left in the outline (no component filter is certified)
        CASE_ID='...'  a lesion found but only partly covered, or a false lump >= 1 cm3
        CASE_ID='...'  a low-contrast scan (the pooled interval is an average-case range there)
""")



